In [12]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition



/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 338, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 338 (delta 40), reused 22 (delta 22), pack-reused 271 (from 2)
Receiving objects: 100% (338/338), 2.70 MiB | 6.87 MiB/s, done.
Resolving deltas: 100% (175/175), done.
/content/vaipe-contextual-pill-recognition


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


tải dataset

In [3]:
!pip install -q kagglehub

import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download("tommyngx/vaipepill2022")

print("Dataset path:", dataset_path)
print("Exists:", Path(dataset_path).exists())

100%|██████████| 25.2G/25.2G [04:25<00:00, 102MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/tommyngx/vaipepill2022/versions/1
Exists: True


In [4]:
from pathlib import Path
import shutil

ZIP_PATH = "/content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip"

print("Zip exists:", Path(ZIP_PATH).exists(), ZIP_PATH)

!rm -rf /content/processed_pika_best
!rm -rf /content/_processed_pika_unzip
!mkdir -p /content/_processed_pika_unzip

!unzip -q "$ZIP_PATH" -d /content/_processed_pika_unzip

tmp = Path("/content/_processed_pika_unzip")

print("Extracted top-level items:")
for p in list(tmp.glob("*"))[:30]:
    print(p)

graph_files = list(tmp.rglob("graph_labels.json"))
print("\nFound graph_labels.json:", graph_files[:5])

if len(graph_files) == 0:
    raise FileNotFoundError("Không tìm thấy graph_labels.json trong processed_pika_best.zip")

src_root = graph_files[0].parent
print("Detected processed root:", src_root)

shutil.copytree(src_root, "/content/processed_pika_best")

print("\nDone.")
print("Data root:", Path("/content/processed_pika_best").exists())
print("Graph labels:", Path("/content/processed_pika_best/graph_labels.json").exists())
print("Graph PMI:", Path("/content/processed_pika_best/graph_pmi.npy").exists())

Zip exists: True /content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip
Extracted top-level items:
/content/_processed_pika_unzip/graph_labels.json
/content/_processed_pika_unzip/graph_pmi.npy
/content/_processed_pika_unzip/pill_crops
/content/_processed_pika_unzip/best_pika_metadata.csv

Found graph_labels.json: [PosixPath('/content/_processed_pika_unzip/graph_labels.json')]
Detected processed root: /content/_processed_pika_unzip

Done.
Data root: True
Graph labels: True
Graph PMI: True


In [5]:
import pandas as pd
from pathlib import Path

csvs = {
    "train": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv",
    "val": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv",
    "test": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv",
}

for name, csv_path in csvs.items():
    df = pd.read_csv(csv_path)

    print("\n====================")
    print("SPLIT:", name)
    print("====================")
    print("CSV rows:", len(df))
    print("Labels:", df["pill_label"].nunique())

    for col in ["pill_crop_path", "prescription_image_path"]:
        exists = df[col].apply(lambda p: Path(str(p)).exists())
        print(f"{col} existing:", int(exists.sum()), "/", len(df))
        print(f"{col} missing:", int((~exists).sum()))

        if (~exists).sum() > 0:
            missing_df = df[~exists]
            show_cols = [c for c in [col, "pill_label", "prescription_key", "source_split"] if c in missing_df.columns]
            print("Missing examples:")
            print(missing_df[show_cols].head(5).to_string(index=False))


SPLIT: train
CSV rows: 23189
Labels: 108
pill_crop_path existing: 23189 / 23189
pill_crop_path missing: 0
prescription_image_path existing: 23189 / 23189
prescription_image_path missing: 0

SPLIT: val
CSV rows: 4974
Labels: 89
pill_crop_path existing: 4974 / 4974
pill_crop_path missing: 0
prescription_image_path existing: 4974 / 4974
prescription_image_path missing: 0

SPLIT: test
CSV rows: 4550
Labels: 82
pill_crop_path existing: 4550 / 4550
pill_crop_path missing: 0
prescription_image_path existing: 4550 / 4550
prescription_image_path missing: 0


######
Tái hiện lại bản baseline PIKA từ đầu trước

M17 sẽ làm nhiệm vụ tái hiện paper

M17 nên có các phần:

1. Dataset split theo prescription_key
2. Graph xây từ train split
3. Graph embedding riêng
4. Pill visual encoder
5. Projection visual feature sang graph embedding space
6. Context attention
7. Pseudo classifier / auxiliary loss
8. Train theo 2 stage nếu có thể

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh build_m17_pika_graph_embeddings.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 250, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 250 (delta 79), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (250/250), 2.59 MiB | 10.72 MiB/s, done.
Resolving deltas: 100% (134/134), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 14K May  4 15:17 build_m17_pika_graph_embeddings.py


In [ ]:
import os

M17_GRAPH_DIR = "/content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts"
os.makedirs(M17_GRAPH_DIR, exist_ok=True)

print("M17 graph output dir:", M17_GRAPH_DIR)

M17 graph output dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python build_m17_pika_graph_embeddings.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --output_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --label_col pill_label \
  --group_col prescription_key \
  --context_col context_labels \
  --embedding_dim 64 \
  --min_cooccur 1 \
  --self_loop_value 1.0 \
  2>&1 | tee /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/build_graph_log.txt

/content/vaipe-contextual-pill-recognition
=== M17 PIKA GRAPH EMBEDDING BUILDER ===
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Output dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Label col: pill_label
Group col: prescription_key
Context col: context_labels
Include context labels: False
Embedding dim: 64
Min cooccur: 1
Self loop value: 1.0

Train rows: 23189
Train groups: 597
Train labels: 108
Num classes: 108
Min label: 0
Max label: 107

Valid prescription groups: 597
Graph cooccur shape: (108, 108)
Non-zero cooccur entries: 1308
Group size distribution:
  1 labels/group: 34 groups
  2 labels/group: 174 groups
  3 labels/group: 200 groups
  4 labels/group: 137 groups
  5 labels/group: 46 groups
  6 labels/group: 6 groups

PMI shape: (108, 108)
PPMI shape: (108, 108)
Embeddings shape: (108, 64)

Saved artifacts:
/content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/build_graph_log.txt
/content/drive/MyDrive/mo

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

out_dir = Path("/content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts")

files = [
    "graph_labels.json",
    "label_to_idx.json",
    "idx_to_label.json",
    "graph_cooccur.npy",
    "graph_pmi.npy",
    "graph_ppmi.npy",
    "graph_embeddings.npy",
    "graph_edges.csv",
    "graph_class_stats.csv",
    "m17_graph_config.json",
]

for f in files:
    p = out_dir / f
    print(f, "=>", p.exists(), p)

labels = json.load(open(out_dir / "graph_labels.json", "r", encoding="utf-8"))
pmi = np.load(out_dir / "graph_pmi.npy")
emb = np.load(out_dir / "graph_embeddings.npy")

print("\nNum labels:", len(labels))
print("PMI shape:", pmi.shape)
print("Embedding shape:", emb.shape)

edge_df = pd.read_csv(out_dir / "graph_edges.csv")
class_df = pd.read_csv(out_dir / "graph_class_stats.csv")

display(edge_df.head(20))
display(class_df.sort_values("instance_count").head(20))

graph_labels.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_labels.json
label_to_idx.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/label_to_idx.json
idx_to_label.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/idx_to_label.json
graph_cooccur.npy => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_cooccur.npy
graph_pmi.npy => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_pmi.npy
graph_ppmi.npy => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_ppmi.npy
graph_embeddings.npy => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
graph_edges.csv => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_edges.csv
graph_class_stats.csv => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_class_stats.csv
m17_graph_config.json => True /content/dr

,src_idx,dst_idx,src_label,dst_label,cooccur_count,pmi,ppmi
0,13,21,13,21,1.0,5.293305,5.293305
1,35,93,35,93,1.0,5.293305,5.293305
2,32,71,32,71,1.0,5.005623,5.005623
3,49,101,49,101,1.0,5.005623,5.005623
4,5,35,5,35,1.0,4.782479,4.782479
5,49,63,49,63,1.0,4.782479,4.782479
6,11,12,11,12,2.0,4.600158,4.600158
7,82,98,82,98,1.0,4.600158,4.600158
8,9,106,9,106,1.0,4.446007,4.446007
9,22,52,22,52,1.0,4.312476,4.312476


,idx,label,instance_count,prescription_doc_count
25,25,25,1,1
49,49,49,1,1
13,13,13,2,1
32,32,32,4,1
88,88,88,6,2
77,77,77,6,2
102,102,102,8,1
66,66,66,8,1
76,76,76,12,2
21,21,21,13,3


In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m17_faithful_pika.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 253 (delta 81), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (253/253), 2.59 MiB | 18.16 MiB/s, done.
Resolving deltas: 100% (136/136), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 25K May  4 15:22 train_m17_faithful_pika.py


In [ ]:
from pathlib import Path

graph_dir = Path("/content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts")

files = [
    "graph_labels.json",
    "label_to_idx.json",
    "idx_to_label.json",
    "graph_embeddings.npy",
    "graph_pmi.npy",
    "m17_graph_config.json",
]

for f in files:
    p = graph_dir / f
    print(f, "=>", p.exists(), p)

graph_labels.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_labels.json
label_to_idx.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/label_to_idx.json
idx_to_label.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/idx_to_label.json
graph_embeddings.npy => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
graph_pmi.npy => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_pmi.npy
m17_graph_config.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/m17_graph_config.json


In [ ]:
import os

M17_TRAIN_DIR = "/content/drive/MyDrive/model/M17_faithful_pika/train_run"
os.makedirs(M17_TRAIN_DIR, exist_ok=True)

print("M17 train output dir:", M17_TRAIN_DIR)

M17 train output dir: /content/drive/MyDrive/model/M17_faithful_pika/train_run


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m17_faithful_pika.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M17_faithful_pika/train_run \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-4 \
  --class_weight_exponent 0.35 \
  --label_smoothing 0.02 \
  --pseudo_loss_weight 0.30 \
  --link_loss_weight 0.10 \
  --dropout_p 0.45 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M17_faithful_pika/train_run/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M17 FAITHFUL PIKA TRAINING ===
Using device: cuda
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV  : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M17_faithful_pika/train_run
Num classes: 108
Graph labels: 108
Graph embeddings shape: (108, 64)
Train pill images existing: 23189 / 23189
Train prescription images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Val prescription images existing: 4974 / 4974

Train rows: 23189
Val rows  : 4974
Train labels: 108
Val labels  : 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64

In [ ]:
!pip install -q kagglehub

import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download("tommyngx/vaipepill2022")

print("Dataset path:", dataset_path)
print("Exists:", Path(dataset_path).exists())

100%|██████████| 25.2G/25.2G [12:07<00:00, 37.2MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/tommyngx/vaipepill2022/versions/1
Exists: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from pathlib import Path
import torch
import pandas as pd

run_dir = Path("/content/drive/MyDrive/model/M17_faithful_pika/train_run")

best_path = run_dir / "M17_faithful_pika_best.pth"
last_path = run_dir / "M17_faithful_pika_last.pth"
hist_path = run_dir / "train_history.csv"

print("Best exists:", best_path.exists(), best_path)
print("Last exists:", last_path.exists(), last_path)
print("History exists:", hist_path.exists(), hist_path)

if best_path.exists():
    best_ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
    print("\nBEST epoch:", best_ckpt.get("epoch"))
    print("BEST val_macro_f1:", best_ckpt.get("val_macro_f1"))

if last_path.exists():
    last_ckpt = torch.load(last_path, map_location="cpu", weights_only=False)
    print("\nLAST epoch:", last_ckpt.get("epoch"))
    print("LAST val_macro_f1:", last_ckpt.get("val_macro_f1"))

if hist_path.exists():
    hist = pd.read_csv(hist_path)
    display(hist.tail(10))

Best exists: True /content/drive/MyDrive/model/M17_faithful_pika/train_run/M17_faithful_pika_best.pth
Last exists: True /content/drive/MyDrive/model/M17_faithful_pika/train_run/M17_faithful_pika_last.pth
History exists: True /content/drive/MyDrive/model/M17_faithful_pika/train_run/train_history.csv

BEST epoch: 9
BEST val_macro_f1: 0.39549658214545763

LAST epoch: 10
LAST val_macro_f1: 0.38342794488233045


,epoch,train_loss,train_main_loss,train_pseudo_loss,train_link_loss,train_acc,train_macro_f1,val_loss,val_acc,val_macro_precision,val_macro_recall,val_macro_f1,lr
0,1,5.280605,4.288255,3.301055,0.020329,0.247057,0.006594,4.385271,0.331524,0.019682,0.021864,0.014268,0.000099
1,2,4.364617,3.621589,2.471021,0.017221,0.409375,0.041818,3.785906,0.530358,0.128141,0.116994,0.099908,0.000096
2,3,3.684720,3.039117,2.146697,0.015946,0.530510,0.105419,3.374181,0.587656,0.170864,0.174504,0.151737,0.000090
3,4,3.151620,2.567622,1.941597,0.015184,0.612446,0.212541,3.086325,0.637515,0.270043,0.240859,0.232300,0.000083
4,5,2.733478,2.193285,1.795769,0.014625,0.682608,0.321668,2.935523,0.648573,0.276824,0.287046,0.268242,0.000075
5,6,2.425302,1.919115,1.682547,0.014229,0.730734,0.406717,2.780773,0.671894,0.324899,0.313210,0.296181,0.000065
6,7,2.175749,1.704943,1.564730,0.013870,0.768856,0.481181,2.814371,0.666064,0.378567,0.337452,0.329071,0.000055
7,8,1.971603,1.533760,1.454972,0.013518,0.797620,0.545894,2.745828,0.686972,0.431984,0.379118,0.376999,0.000045
8,9,1.810748,1.403300,1.353774,0.013164,0.828755,0.604412,2.815663,0.676317,0.426573,0.406804,0.395497,0.000035
9,10,1.672012,1.293238,1.258297,0.012850,0.853034,0.638545,2.800467,0.674507,0.408512,0.394039,0.383428,0.000025


In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m17_faithful_pika.py train_m17_faithful_pika_resume.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 258 (delta 83), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (258/258), 2.59 MiB | 6.78 MiB/s, done.
Resolving deltas: 100% (138/138), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 25K May  5 06:15 train_m17_faithful_pika.py
-rw-r--r-- 1 root root 15K May  5 06:15 train_m17_faithful_pika_resume.py


In [ ]:
from pathlib import Path

graph_dir = Path("/content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts")

files = [
    "graph_labels.json",
    "label_to_idx.json",
    "idx_to_label.json",
    "graph_embeddings.npy",
    "graph_pmi.npy",
]

for f in files:
    p = graph_dir / f
    print(f, "=>", p.exists(), p)

graph_labels.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_labels.json
label_to_idx.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/label_to_idx.json
idx_to_label.json => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/idx_to_label.json
graph_embeddings.npy => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
graph_pmi.npy => True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_pmi.npy


In [ ]:
from pathlib import Path
import shutil

ZIP_PATH = "/content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip"

print("Zip exists:", Path(ZIP_PATH).exists(), ZIP_PATH)

!rm -rf /content/processed_pika_best
!rm -rf /content/_processed_pika_unzip
!mkdir -p /content/_processed_pika_unzip

!unzip -q "$ZIP_PATH" -d /content/_processed_pika_unzip

tmp = Path("/content/_processed_pika_unzip")

print("Extracted top-level items:")
for p in list(tmp.glob("*"))[:30]:
    print(p)

# Tìm thư mục chứa graph_labels.json để xác định root đúng
graph_files = list(tmp.rglob("graph_labels.json"))
print("\nFound graph_labels.json:", graph_files[:5])

if len(graph_files) == 0:
    raise FileNotFoundError("Không tìm thấy graph_labels.json trong processed_pika_best.zip")

src_root = graph_files[0].parent
print("Detected processed root:", src_root)

shutil.copytree(src_root, "/content/processed_pika_best")

print("\nDone.")
print("Data root:", Path("/content/processed_pika_best").exists())
print("Graph labels:", Path("/content/processed_pika_best/graph_labels.json").exists())
print("Graph PMI:", Path("/content/processed_pika_best/graph_pmi.npy").exists())

Zip exists: True /content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip
error:  zipfile read error
Extracted top-level items:
/content/_processed_pika_unzip/pill_crops
/content/_processed_pika_unzip/best_pika_metadata.csv
/content/_processed_pika_unzip/graph_pmi.npy
/content/_processed_pika_unzip/graph_labels.json

Found graph_labels.json: [PosixPath('/content/_processed_pika_unzip/graph_labels.json')]
Detected processed root: /content/_processed_pika_unzip

Done.
Data root: True
Graph labels: True
Graph PMI: True


In [ ]:
import pandas as pd
from pathlib import Path

csvs = {
    "train": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv",
    "val": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv",
    "test": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv",
}

for name, csv_path in csvs.items():
    df = pd.read_csv(csv_path)

    print("\n====================")
    print("SPLIT:", name)
    print("====================")
    print("CSV rows:", len(df))
    print("Labels:", df["pill_label"].nunique())

    for col in ["pill_crop_path", "prescription_image_path"]:
        exists = df[col].apply(lambda p: Path(str(p)).exists())
        print(f"{col} existing:", int(exists.sum()), "/", len(df))
        print(f"{col} missing:", int((~exists).sum()))


SPLIT: train
CSV rows: 23189
Labels: 108
pill_crop_path existing: 15720 / 23189
pill_crop_path missing: 7469
prescription_image_path existing: 23189 / 23189
prescription_image_path missing: 0

SPLIT: val
CSV rows: 4974
Labels: 89
pill_crop_path existing: 3282 / 4974
pill_crop_path missing: 1692
prescription_image_path existing: 4974 / 4974
prescription_image_path missing: 0

SPLIT: test
CSV rows: 4550
Labels: 82
pill_crop_path existing: 3149 / 4550
pill_crop_path missing: 1401
prescription_image_path existing: 4550 / 4550
prescription_image_path missing: 0


In [ ]:
import os

RESUME_DIR = "/content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9"
os.makedirs(RESUME_DIR, exist_ok=True)

print("Resume output dir:", RESUME_DIR)

Resume output dir: /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m17_faithful_pika_resume.py \
  --resume_checkpoint /content/drive/MyDrive/model/M17_faithful_pika/train_run/M17_faithful_pika_best.pth \
  --previous_history_csv /content/drive/MyDrive/model/M17_faithful_pika/train_run/train_history.csv \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9 \
  --total_epochs 15 \
  --batch_size 16 \
  --lr 4.5e-5 \
  --class_weight_exponent 0.35 \
  --label_smoothing 0.02 \
  --pseudo_loss_weight 0.30 \
  --link_loss_weight 0.10 \
  --dropout_p 0.45 \
  --patience 4 \
  2>&1 | tee /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M17 FAITHFUL PIKA RESUME TRAINING ===
Using device: cuda
Resume checkpoint: /content/drive/MyDrive/model/M17_faithful_pika/train_run/M17_faithful_pika_best.pth
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV  : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9

Resume epoch: 9
Start epoch : 10
Initial best Val Macro F1: 0.39549658214545763
Num classes: 108
Graph dim: 64
Max context len: 5
Graph embeddings shape: (108, 64)
Train pill images existing: 23189 / 23189
Train prescription images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Val prescription images existing: 4974 / 4974

Train rows: 2

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh evaluate_m17_faithful_pika.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 261, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 261 (delta 85), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (261/261), 2.60 MiB | 6.66 MiB/s, done.
Resolving deltas: 100% (140/140), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 13K May  5 07:56 evaluate_m17_faithful_pika.py


In [ ]:
import os

EVAL_DIR = "/content/drive/MyDrive/model/M17_faithful_pika/test_eval"
os.makedirs(EVAL_DIR, exist_ok=True)

print("Eval output dir:", EVAL_DIR)

Eval output dir: /content/drive/MyDrive/model/M17_faithful_pika/test_eval


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python evaluate_m17_faithful_pika.py \
  --checkpoint /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/M17_faithful_pika_resume_best.pth \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M17_faithful_pika/test_eval \
  --batch_size 32 \
  2>&1 | tee /content/drive/MyDrive/model/M17_faithful_pika/test_eval/test_log.txt

/content/vaipe-contextual-pill-recognition
=== EVALUATE M17 FAITHFUL PIKA ===
Using device: cuda
Checkpoint: /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/M17_faithful_pika_resume_best.pth
Test CSV  : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M17_faithful_pika/test_eval
Using graph embeddings: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
Num classes: 108
Max context len: 5
Graph embeddings shape: (108, 64)

Raw test rows: 4550
Raw test labels: 82
Raw test columns: ['pill_crop_path', 'pill_label', 'pill_json', 'prescription_json', 'prescription_image_path', 'context_labels', 'prescription_key', 'split', 'source_split', 'clean_split']
Test pill images existing: 4550 / 4550
Test prescription images existing: 4550 / 4550

Test rows after image check: 4550
Test la

đây là baseline chính thức:

M17 faithful PIKA baseline:
Test Macro F1 Present = 0.3941

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m18_improved_pika.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 264, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 264 (delta 87), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (264/264), 2.60 MiB | 6.37 MiB/s, done.
Resolving deltas: 100% (142/142), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 17K May  5 08:02 train_m18_improved_pika.py


In [ ]:
import os

M18_DIR = "/content/drive/MyDrive/model/M18_improved_pika/train_run"
os.makedirs(M18_DIR, exist_ok=True)

print("M18 output dir:", M18_DIR)

M18 output dir: /content/drive/MyDrive/model/M18_improved_pika/train_run


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m18_improved_pika.py \
  --base_checkpoint /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/M17_faithful_pika_resume_best.pth \
  --previous_history_csv /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/train_history.csv \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M18_improved_pika/train_run \
  --epochs 8 \
  --batch_size 16 \
  --lr 2.5e-5 \
  --class_weight_exponent 0.35 \
  --label_smoothing 0.02 \
  --pseudo_loss_weight 0.15 \
  --link_loss_weight 0.05 \
  --dropout_p 0.45 \
  --train_graph_embeddings \
  --patience 4 \
  --min_delta 0.0001 \
  2>&1 | tee /content/drive/MyDrive/model/M18_improved_pika/train_run/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M18 IMPROVED PIKA TRAINING ===
Using device: cuda
Base checkpoint: /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/M17_faithful_pika_resume_best.pth
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV  : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M18_improved_pika/train_run

Base epoch: 15
Base Val Macro F1: 0.4175186514209033
Num classes: 108
Graph dim: 64
Max context len: 5
Pill model: tf_efficientnetv2_s.in21k_ft_in1k
Prescription model: resnet18.a1_in1k

Graph embeddings shape: (108, 64)
Train pill images existing: 23189 / 23189
Train prescription images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Val p

In [ ]:
import os

M18_V2_DIR = "/content/drive/MyDrive/model/M18_improved_pika/train_run_v2_safe"
os.makedirs(M18_V2_DIR, exist_ok=True)

print("M18 v2 output dir:", M18_V2_DIR)

M18 v2 output dir: /content/drive/MyDrive/model/M18_improved_pika/train_run_v2_safe


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m18_improved_pika.py \
  --base_checkpoint /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/M17_faithful_pika_resume_best.pth \
  --previous_history_csv /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/train_history.csv \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M18_improved_pika/train_run_v2_safe \
  --epochs 6 \
  --batch_size 16 \
  --lr 1e-5 \
  --class_weight_exponent 0.35 \
  --label_smoothing 0.02 \
  --pseudo_loss_weight 0.20 \
  --link_loss_weight 0.05 \
  --dropout_p 0.50 \
  --freeze_backbone_epochs 1 \
  --patience 3 \
  --min_delta 0.0001 \
  2>&1 | tee /content/drive/MyDrive/model/M18_improved_pika/train_run_v2_safe/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M18 IMPROVED PIKA TRAINING ===
Using device: cuda
Base checkpoint: /content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/M17_faithful_pika_resume_best.pth
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV  : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M18_improved_pika/train_run_v2_safe

Base epoch: 15
Base Val Macro F1: 0.4175186514209033
Num classes: 108
Graph dim: 64
Max context len: 5
Pill model: tf_efficientnetv2_s.in21k_ft_in1k
Prescription model: resnet18.a1_in1k

Graph embeddings shape: (108, 64)
Train pill images existing: 23189 / 23189
Train prescription images existing: 23189 / 23189
Val pill images existing: 4974 / 49

Sau khi M18_v2 chạy xong, dùng lại evaluate_m17_faithful_pika.py để đánh giá checkpoint M18 trên test, vì model class vẫn là M17FaithfulPIKAModel.

In [ ]:
import os

M18_EVAL_DIR = "/content/drive/MyDrive/model/M18_improved_pika/test_eval_v2_safe"
os.makedirs(M18_EVAL_DIR, exist_ok=True)

print("M18 eval output dir:", M18_EVAL_DIR)

M18 eval output dir: /content/drive/MyDrive/model/M18_improved_pika/test_eval_v2_safe


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python evaluate_m17_faithful_pika.py \
  --checkpoint /content/drive/MyDrive/model/M18_improved_pika/train_run_v2_safe/M18_improved_pika_best.pth \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M18_improved_pika/test_eval_v2_safe \
  --batch_size 32 \
  2>&1 | tee /content/drive/MyDrive/model/M18_improved_pika/test_eval_v2_safe/test_log.txt

/content/vaipe-contextual-pill-recognition
=== EVALUATE M17 FAITHFUL PIKA ===
Using device: cuda
Checkpoint: /content/drive/MyDrive/model/M18_improved_pika/train_run_v2_safe/M18_improved_pika_best.pth
Test CSV  : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M18_improved_pika/test_eval_v2_safe
Using graph embeddings: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
Num classes: 108
Max context len: 5
Graph embeddings shape: (108, 64)

Raw test rows: 4550
Raw test labels: 82
Raw test columns: ['pill_crop_path', 'pill_label', 'pill_json', 'prescription_json', 'prescription_image_path', 'context_labels', 'prescription_key', 'split', 'source_split', 'clean_split']
Test pill images existing: 4550 / 4550
Test prescription images existing: 4550 / 4550

Test rows after image check: 4550
Test labels present

M19 vẫn giữ nền tảng PIKA:

pill image
prescription image
graph embedding
context labels

Nhưng cải tiến phần fusion/context:

M17 fusion:
[pill_z, pres_z, context_z] → classifier

M19 fusion:
[pill_z, pres_z, context_z,
 pill_z * context_z,
 pill_z * pres_z,
 |pill_z - context_z|,
 |pill_z - pres_z|]
→ deeper classifier

Thêm nữa:

context dropout

để model không quá phụ thuộc vào context/prescription và giảm overfit.

In [ ]:
!pip install -q timm kagglehub scikit-learn pandas numpy tqdm pillow

In [ ]:
from pathlib import Path

paths = {
    "Drive model folder": "/content/drive/MyDrive/model",
    "Processed zip": "/content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip",

    "Split v2 folder": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2",
    "Train clean v2": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv",
    "Val clean v2": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv",
    "Test clean v2": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv",

    "M17 graph dir": "/content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts",
    "M17 graph embeddings": "/content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy",
    "M17 label_to_idx": "/content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/label_to_idx.json",
    "M17 idx_to_label": "/content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/idx_to_label.json",

    "M17 best ckpt": "/content/drive/MyDrive/model/M17_faithful_pika/train_run_resume_from_best_ep9/M17_faithful_pika_resume_best.pth",
    "M18 v2 best ckpt": "/content/drive/MyDrive/model/M18_improved_pika/train_run_v2_safe/M18_improved_pika_best.pth",
}

for name, p in paths.items():
    print(f"{name:24}:", Path(p).exists(), p)

Drive model folder      : True /content/drive/MyDrive/model
Processed zip           : True /content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip
Split v2 folder         : True /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2
Train clean v2          : True /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val clean v2            : True /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test clean v2           : True /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
M17 graph dir           : True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
M17 graph embeddings    : True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
M17 label_to_idx        : True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/label_to_idx.json
M17 idx_to_label        : True /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/idx_to_labe

In [ ]:
from pathlib import Path

sample_paths = [
    "/root/.cache/kagglehub/datasets/tommyngx/vaipepill2022/versions/1/public_train/prescription/image/VAIPE_P_TRAIN_188.png",
    "/root/.cache/kagglehub/datasets/tommyngx/vaipepill2022/versions/1/public_train/prescription/image/VAIPE_P_TRAIN_711.png",
]

for p in sample_paths:
    print(Path(p).exists(), p)

True /root/.cache/kagglehub/datasets/tommyngx/vaipepill2022/versions/1/public_train/prescription/image/VAIPE_P_TRAIN_188.png
True /root/.cache/kagglehub/datasets/tommyngx/vaipepill2022/versions/1/public_train/prescription/image/VAIPE_P_TRAIN_711.png


In [ ]:
from pathlib import Path
import shutil

ZIP_PATH = "/content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip"

print("Zip exists:", Path(ZIP_PATH).exists(), ZIP_PATH)

!rm -rf /content/processed_pika_best
!rm -rf /content/_processed_pika_unzip
!mkdir -p /content/_processed_pika_unzip

!unzip -q "$ZIP_PATH" -d /content/_processed_pika_unzip

tmp = Path("/content/_processed_pika_unzip")

print("Extracted top-level items:")
for p in list(tmp.glob("*"))[:30]:
    print(p)

graph_files = list(tmp.rglob("graph_labels.json"))
print("\nFound graph_labels.json:", graph_files[:5])

if len(graph_files) == 0:
    raise FileNotFoundError("Không tìm thấy graph_labels.json trong processed_pika_best.zip")

src_root = graph_files[0].parent
print("Detected processed root:", src_root)

shutil.copytree(src_root, "/content/processed_pika_best")

print("\nDone.")
print("Data root:", Path("/content/processed_pika_best").exists())
print("Graph labels:", Path("/content/processed_pika_best/graph_labels.json").exists())
print("Graph PMI:", Path("/content/processed_pika_best/graph_pmi.npy").exists())

Zip exists: True /content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip
Extracted top-level items:
/content/_processed_pika_unzip/pill_crops
/content/_processed_pika_unzip/best_pika_metadata.csv
/content/_processed_pika_unzip/graph_pmi.npy
/content/_processed_pika_unzip/graph_labels.json

Found graph_labels.json: [PosixPath('/content/_processed_pika_unzip/graph_labels.json')]
Detected processed root: /content/_processed_pika_unzip

Done.
Data root: True
Graph labels: True
Graph PMI: True


In [ ]:
import pandas as pd
from pathlib import Path

csvs = {
    "train": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv",
    "val": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv",
    "test": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv",
}

for name, csv_path in csvs.items():
    df = pd.read_csv(csv_path)

    print("\n====================")
    print("SPLIT:", name)
    print("====================")
    print("CSV rows:", len(df))
    print("Labels:", df["pill_label"].nunique())

    for col in ["pill_crop_path", "prescription_image_path"]:
        exists = df[col].apply(lambda p: Path(str(p)).exists())
        print(f"{col} existing:", int(exists.sum()), "/", len(df))
        print(f"{col} missing:", int((~exists).sum()))

        if (~exists).sum() > 0:
            missing_df = df[~exists]
            show_cols = [c for c in [col, "pill_label", "prescription_key", "source_split"] if c in missing_df.columns]
            print("Missing examples:")
            print(missing_df[show_cols].head(5).to_string(index=False))


SPLIT: train
CSV rows: 23189
Labels: 108
pill_crop_path existing: 23189 / 23189
pill_crop_path missing: 0
prescription_image_path existing: 23189 / 23189
prescription_image_path missing: 0

SPLIT: val
CSV rows: 4974
Labels: 89
pill_crop_path existing: 4974 / 4974
pill_crop_path missing: 0
prescription_image_path existing: 4974 / 4974
prescription_image_path missing: 0

SPLIT: test
CSV rows: 4550
Labels: 82
pill_crop_path existing: 4550 / 4550
pill_crop_path missing: 0
prescription_image_path existing: 4550 / 4550
prescription_image_path missing: 0


In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m17_faithful_pika.py
!ls -lh train_m18_improved_pika.py
!ls -lh evaluate_m17_faithful_pika.py
!ls -lh train_m19_arch_pika_v1.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 267, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 267 (delta 89), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (267/267), 2.60 MiB | 10.21 MiB/s, done.
Resolving deltas: 100% (144/144), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 25K May  5 10:58 train_m17_faithful_pika.py
-rw-r--r-- 1 root root 17K May  5 10:58 train_m18_improved_pika.py
-rw-r--r-- 1 root root 13K May  5 10:58 evaluate_m17_faithful_pika.py
-rw-r--r-- 1 root root 26K May  5 10:58 train_m19_arch_pika_v1.py


In [ ]:
import os

M19_DIR = "/content/drive/MyDrive/model/M19_arch_pika_v1/train_run"
os.makedirs(M19_DIR, exist_ok=True)

print("M19 output dir:", M19_DIR)

M19 output dir: /content/drive/MyDrive/model/M19_arch_pika_v1/train_run


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m19_arch_pika_v1.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M19_arch_pika_v1/train_run \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-4 \
  --class_weight_exponent 0.35 \
  --label_smoothing 0.02 \
  --pseudo_loss_weight 0.20 \
  --link_loss_weight 0.05 \
  --dropout_p 0.50 \
  --context_dropout_p 0.20 \
  --freeze_backbone_epochs 1 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M19_arch_pika_v1/train_run/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M19 ARCHITECTURE PIKA V1 TRAINING ===
Using device: cuda
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M19_arch_pika_v1/train_run
Num classes: 108
Graph embeddings shape: (108, 64)
Train pill images existing: 23189 / 23189
Train prescription images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Val prescription images existing: 4974 / 4974
Train rows: 23189
Val rows: 4974
Train labels: 108
Val labels: 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64(35), np.int64(44), 

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh evaluate_m19_arch_pika_v1.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 270 (delta 91), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (270/270), 2.61 MiB | 17.67 MiB/s, done.
Resolving deltas: 100% (146/146), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 14K May  5 14:28 evaluate_m19_arch_pika_v1.py


In [ ]:
import os

M19_EVAL_DIR = "/content/drive/MyDrive/model/M19_arch_pika_v1/test_eval"
os.makedirs(M19_EVAL_DIR, exist_ok=True)

print("M19 eval output dir:", M19_EVAL_DIR)

M19 eval output dir: /content/drive/MyDrive/model/M19_arch_pika_v1/test_eval


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python evaluate_m19_arch_pika_v1.py \
  --checkpoint /content/drive/MyDrive/model/M19_arch_pika_v1/train_run/M19_arch_pika_v1_best.pth \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M19_arch_pika_v1/test_eval \
  --batch_size 32 \
  2>&1 | tee /content/drive/MyDrive/model/M19_arch_pika_v1/test_eval/test_log.txt

/content/vaipe-contextual-pill-recognition
=== EVALUATE M19 ARCHITECTURE PIKA V1 ===
Using device: cuda
Checkpoint: /content/drive/MyDrive/model/M19_arch_pika_v1/train_run/M19_arch_pika_v1_best.pth
Test CSV  : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M19_arch_pika_v1/test_eval
Using graph embeddings: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
Num classes: 108
Max context len: 5
Graph embeddings shape: (108, 64)

Raw test rows: 4550
Raw test labels: 82
Raw test columns: ['pill_crop_path', 'pill_label', 'pill_json', 'prescription_json', 'prescription_image_path', 'context_labels', 'prescription_key', 'split', 'source_split', 'clean_split']
Test pill images existing: 4550 / 4550
Test prescription images existing: 4550 / 4550

Test rows after image check: 4550
Test labels present: 82
Test ma

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh build_m20_pruned_graph_embeddings.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 273, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 273 (delta 92), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (273/273), 2.61 MiB | 18.69 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 7.2K May  5 14:36 build_m20_pruned_graph_embeddings.py


Build graph prune 20%

In [ ]:
import os

M20_GRAPH_DIR = "/content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts"
os.makedirs(M20_GRAPH_DIR, exist_ok=True)

print("M20 graph output dir:", M20_GRAPH_DIR)

M20 graph output dir: /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python build_m20_pruned_graph_embeddings.py \
  --base_graph_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts \
  --prune_ratio 0.20 \
  --embedding_dim 64 \
  2>&1 | tee /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/build_log.txt

/content/vaipe-contextual-pill-recognition
=== M20 PRUNED GRAPH EMBEDDING BUILDER ===
Base graph dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts
Prune ratio: 0.2
Embedding dim: 64
Loaded PPMI: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_ppmi.npy
PPMI shape: (108, 108)
Num labels: 108

Pruning stats:
total_positive_offdiag_edges: 926
removed_edges: 186
remaining_positive_offdiag_edges: 740
threshold: 0.4838341772556305
prune_ratio: 0.2
keep_diagonal: True

Pruned PPMI shape: (108, 108)
Embeddings shape: (108, 64)

Saved artifacts:
/content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/build_log.txt
/content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/graph_edges_pruned.csv
/content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/graph_embeddings.npy
/content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_arti

In [ ]:
from pathlib import Path
import numpy as np
import json
import pandas as pd

gdir = Path("/content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts")

files = [
    "graph_labels.json",
    "label_to_idx.json",
    "idx_to_label.json",
    "graph_pmi.npy",
    "graph_ppmi.npy",
    "graph_embeddings.npy",
    "graph_edges_pruned.csv",
    "m20_pruned_graph_config.json",
]

for f in files:
    p = gdir / f
    print(f, "=>", p.exists(), p)

emb = np.load(gdir / "graph_embeddings.npy")
ppmi = np.load(gdir / "graph_ppmi.npy")
labels = json.load(open(gdir / "graph_labels.json", "r", encoding="utf-8"))

print("Num labels:", len(labels))
print("PPMI shape:", ppmi.shape)
print("Embedding shape:", emb.shape)

edge_df = pd.read_csv(gdir / "graph_edges_pruned.csv")
display(edge_df.head(20))

graph_labels.json => True /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/graph_labels.json
label_to_idx.json => True /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/label_to_idx.json
idx_to_label.json => True /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/idx_to_label.json
graph_pmi.npy => True /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/graph_pmi.npy
graph_ppmi.npy => True /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/graph_ppmi.npy
graph_embeddings.npy => True /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/graph_embeddings.npy
graph_edges_pruned.csv => True /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/graph_edges_pruned.csv
m20_pruned_graph_config.json => True /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts/m20_pruned_graph_config.json
Num labels: 108
PPMI shape: (108, 108)
Embedding shape: 

,src_idx,dst_idx,src_label,dst_label,weight
0,13,21,13,21,5.293305
1,35,93,35,93,5.293305
2,32,71,32,71,5.005623
3,49,101,49,101,5.005623
4,49,63,49,63,4.782479
5,5,35,5,35,4.782479
6,11,12,11,12,4.600158
7,82,98,82,98,4.600158
8,9,106,9,106,4.446007
9,22,52,22,52,4.312476


In [ ]:
import os

M20_TRAIN_DIR = "/content/drive/MyDrive/model/M20_graph_pruning/train_prune20"
os.makedirs(M20_TRAIN_DIR, exist_ok=True)

print("M20 train output dir:", M20_TRAIN_DIR)

M20 train output dir: /content/drive/MyDrive/model/M20_graph_pruning/train_prune20


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m19_arch_pika_v1.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M20_graph_pruning/train_prune20 \
  --best_name M20_prune20_best.pth \
  --last_name M20_prune20_last.pth \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-4 \
  --class_weight_exponent 0.35 \
  --label_smoothing 0.02 \
  --pseudo_loss_weight 0.20 \
  --link_loss_weight 0.05 \
  --dropout_p 0.50 \
  --context_dropout_p 0.20 \
  --freeze_backbone_epochs 1 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M20_graph_pruning/train_prune20/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M19 ARCHITECTURE PIKA V1 TRAINING ===
Using device: cuda
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M20_graph_pruning/prune20_graph_artifacts
Output dir: /content/drive/MyDrive/model/M20_graph_pruning/train_prune20
Num classes: 108
Graph embeddings shape: (108, 64)
Train pill images existing: 23189 / 23189
Train prescription images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Val prescription images existing: 4974 / 4974
Train rows: 23189
Val rows: 4974
Train labels: 108
Val labels: 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64(35), n

#Prune 10%

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install -q kagglehub

import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download("tommyngx/vaipepill2022")

print("Dataset path:", dataset_path)
print("Exists:", Path(dataset_path).exists())

100%|██████████| 25.2G/25.2G [23:54<00:00, 18.9MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/tommyngx/vaipepill2022/versions/1
Exists: True


In [ ]:
from pathlib import Path
import shutil

ZIP_PATH = "/content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip"

print("Zip exists:", Path(ZIP_PATH).exists(), ZIP_PATH)

!rm -rf /content/processed_pika_best
!rm -rf /content/_processed_pika_unzip
!mkdir -p /content/_processed_pika_unzip

!unzip -q "$ZIP_PATH" -d /content/_processed_pika_unzip

tmp = Path("/content/_processed_pika_unzip")

print("Extracted top-level items:")
for p in list(tmp.glob("*"))[:30]:
    print(p)

graph_files = list(tmp.rglob("graph_labels.json"))
print("\nFound graph_labels.json:", graph_files[:5])

if len(graph_files) == 0:
    raise FileNotFoundError("Không tìm thấy graph_labels.json trong processed_pika_best.zip")

src_root = graph_files[0].parent
print("Detected processed root:", src_root)

shutil.copytree(src_root, "/content/processed_pika_best")

print("\nDone.")
print("Data root:", Path("/content/processed_pika_best").exists())
print("Graph labels:", Path("/content/processed_pika_best/graph_labels.json").exists())
print("Graph PMI:", Path("/content/processed_pika_best/graph_pmi.npy").exists())

Zip exists: True /content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip
error:  zipfile read error
Extracted top-level items:
/content/_processed_pika_unzip/pill_crops
/content/_processed_pika_unzip/best_pika_metadata.csv
/content/_processed_pika_unzip/graph_pmi.npy
/content/_processed_pika_unzip/graph_labels.json

Found graph_labels.json: [PosixPath('/content/_processed_pika_unzip/graph_labels.json')]
Detected processed root: /content/_processed_pika_unzip

Done.
Data root: True
Graph labels: True
Graph PMI: True


In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 273, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 273 (delta 92), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (273/273), 2.61 MiB | 12.09 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/vaipe-contextual-pill-recognition
total 4.3M
-rw-r--r-- 1 root root 9.0K May  6 10:56  analyze_m13_errors.py
-rw-r--r-- 1 root root 7.6K May  6 10:56  build_best_pika_data.py
-rw-r--r-- 1 root root  14K May  6 10:56  build_m17_pika_graph_embeddings.py
-rw-r--r-- 1 root root 7.2K May  6 10:56  build_m20_pruned_graph_embeddings.py
-rw-r--r-- 1 root root 5.4K May  6 10:56  build_pika_context_metadata.py
-rw-r--r-- 1 root root 6.6K May  6 10:56  build_pika_graph_data.py
-rw-r--r-- 1 root root 4.4K May  6 10:56  build_pika_metadata.py
-rw-r--r-- 1 root root 5.7K May  6 10:56  build_pika_v3_metadata.py
-rw-r

In [ ]:
!ls -lh build_m20_pruned_graph_embeddings.py train_m19_arch_pika_v1.py

-rw-r--r-- 1 root root 7.2K May  6 10:56 build_m20_pruned_graph_embeddings.py
-rw-r--r-- 1 root root  26K May  6 10:56 train_m19_arch_pika_v1.py


In [ ]:
import pandas as pd
from pathlib import Path

train_csv = "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv"
val_csv = "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv"

for name, csv_path in [("train", train_csv), ("val", val_csv)]:
    df = pd.read_csv(csv_path)
    df["pill_exists"] = df["pill_crop_path"].apply(lambda p: Path(str(p)).exists())
    df["pres_exists"] = df["prescription_image_path"].apply(lambda p: Path(str(p)).exists())

    print("\n===", name, "===")
    print("rows:", len(df))
    print("pill existing:", int(df["pill_exists"].sum()), "/", len(df))
    print("pres existing:", int(df["pres_exists"].sum()), "/", len(df))

    missing = df[~df["pill_exists"]]
    print("\nMissing pill examples:")
    print(missing[["pill_crop_path", "pill_label", "prescription_key", "source_split"]].head(10).to_string(index=False))

    if "source_split" in df.columns:
        print("\nPill exists by source_split:")
        display(
            df.groupby("source_split")["pill_exists"]
            .agg(["count", "sum"])
            .assign(missing=lambda x: x["count"] - x["sum"])
        )


=== train ===
rows: 23189
pill existing: 9647 / 23189
pres existing: 23189 / 23189

Missing pill examples:
                                               pill_crop_path  pill_label prescription_key source_split
 /content/processed_pika_best/pill_crops/VAIPE_P_0_0_box1.jpg          64  VAIPE_P_TRAIN_0    old_train
 /content/processed_pika_best/pill_crops/VAIPE_P_0_1_box0.jpg          64  VAIPE_P_TRAIN_0    old_train
 /content/processed_pika_best/pill_crops/VAIPE_P_0_1_box1.jpg          64  VAIPE_P_TRAIN_0    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_0_10_box1.jpg          64  VAIPE_P_TRAIN_0    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_0_11_box0.jpg          64  VAIPE_P_TRAIN_0    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_0_12_box0.jpg          64  VAIPE_P_TRAIN_0    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_0_12_box1.jpg          64  VAIPE_P_TRAIN_0    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_0_12_box2.jp

,count,sum,missing
source_split,,,
old_test,4024,1676,2348
old_train,15785,6555,9230
old_val,3380,1416,1964



=== val ===
rows: 4974
pill existing: 2004 / 4974
pres existing: 4974 / 4974

Missing pill examples:
                                                 pill_crop_path  pill_label   prescription_key source_split
/content/processed_pika_best/pill_crops/VAIPE_P_1011_0_box0.jpg          19 VAIPE_P_TRAIN_1011    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_1011_0_box3.jpg          91 VAIPE_P_TRAIN_1011    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_1011_1_box0.jpg         104 VAIPE_P_TRAIN_1011    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_1011_1_box1.jpg          19 VAIPE_P_TRAIN_1011    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_1011_1_box2.jpg          71 VAIPE_P_TRAIN_1011    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_1011_1_box3.jpg          91 VAIPE_P_TRAIN_1011    old_train
/content/processed_pika_best/pill_crops/VAIPE_P_1011_2_box1.jpg         104 VAIPE_P_TRAIN_1011    old_train
/content/processed_pika_best/pill_

,count,sum,missing
source_split,,,
old_test,1128,488,640
old_train,3246,1270,1976
old_val,600,246,354


In [ ]:
from pathlib import Path

root = Path("/content/processed_pika_best")

print("processed exists:", root.exists())
print("graph_labels:", (root / "graph_labels.json").exists())
print("graph_pmi:", (root / "graph_pmi.npy").exists())

if root.exists():
    print("\nTop-level folders/files:")
    for p in list(root.iterdir())[:50]:
        print(p)

    print("\nTotal files:")
    !find /content/processed_pika_best -type f | wc -l

    print("\nExample crop files:")
    !find /content/processed_pika_best -type f \( -iname "*.png" -o -iname "*.jpg" -o -iname "*.jpeg" \) | head -30

processed exists: True
graph_labels: True
graph_pmi: True

Top-level folders/files:
/content/processed_pika_best/pill_crops
/content/processed_pika_best/best_pika_metadata.csv
/content/processed_pika_best/graph_pmi.npy
/content/processed_pika_best/graph_labels.json

Total files:
13621

Example crop files:
/content/processed_pika_best/pill_crops/VAIPE_P_1092_2_box1.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_872_5_box0.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_595_5_box0.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_180_9_box0.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_482_21_box1.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_247_19_box1.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_233_0_box1.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_353_33_box3.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_107_1_box0.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_809_0_box3.jpg
/content/processed_pika_best/pill_crops/VAIPE_P_893_5_box5.

In [ ]:
from pathlib import Path
import shutil

ZIP_PATH = "/content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip"

print("Zip exists:", Path(ZIP_PATH).exists(), ZIP_PATH)
print("Zip size GB:", round(Path(ZIP_PATH).stat().st_size / 1024**3, 3) if Path(ZIP_PATH).exists() else None)

!rm -rf /content/processed_pika_best
!rm -rf /content/_processed_pika_unzip
!mkdir -p /content/_processed_pika_unzip

!unzip -q "$ZIP_PATH" -d /content/_processed_pika_unzip

tmp = Path("/content/_processed_pika_unzip")

print("\nExtracted top-level items:")
for p in list(tmp.glob("*"))[:30]:
    print(p)

graph_files = list(tmp.rglob("graph_labels.json"))
print("\nFound graph_labels.json:")
for p in graph_files[:10]:
    print(p)

if len(graph_files) == 0:
    raise FileNotFoundError("Không tìm thấy graph_labels.json trong processed_pika_best.zip")

# Chọn root chứa graph_labels.json và nhiều file nhất
candidates = []
for gf in graph_files:
    parent = gf.parent
    num_files = sum(1 for _ in parent.rglob("*") if _.is_file())
    candidates.append((num_files, parent))

candidates = sorted(candidates, reverse=True)
src_root = candidates[0][1]

print("\nDetected processed root:", src_root)
print("Files under detected root:", candidates[0][0])

shutil.copytree(src_root, "/content/processed_pika_best")

print("\nDone.")
print("Data root:", Path("/content/processed_pika_best").exists())
print("Graph labels:", Path("/content/processed_pika_best/graph_labels.json").exists())
print("Graph PMI:", Path("/content/processed_pika_best/graph_pmi.npy").exists())

print("\nTotal files after restore:")
!find /content/processed_pika_best -type f | wc -l

Zip exists: True /content/drive/MyDrive/VAIPE_FINAL_BACKUP/processed_pika_best.zip
Zip size GB: 0.652

Extracted top-level items:
/content/_processed_pika_unzip/pill_crops
/content/_processed_pika_unzip/best_pika_metadata.csv
/content/_processed_pika_unzip/graph_pmi.npy
/content/_processed_pika_unzip/graph_labels.json

Found graph_labels.json:
/content/_processed_pika_unzip/graph_labels.json

Detected processed root: /content/_processed_pika_unzip
Files under detected root: 32716

Done.
Data root: True
Graph labels: True
Graph PMI: True

Total files after restore:
32716


In [ ]:
import pandas as pd
from pathlib import Path

csvs = {
    "train": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv",
    "val": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv",
    "test": "/content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv",
}

for name, csv_path in csvs.items():
    df = pd.read_csv(csv_path)

    print("\n====================")
    print("SPLIT:", name)
    print("====================")
    print("CSV rows:", len(df))
    print("Labels:", df["pill_label"].nunique())

    for col in ["pill_crop_path", "prescription_image_path"]:
        exists = df[col].apply(lambda p: Path(str(p)).exists())
        print(f"{col} existing:", int(exists.sum()), "/", len(df))
        print(f"{col} missing:", int((~exists).sum()))


SPLIT: train
CSV rows: 23189
Labels: 108
pill_crop_path existing: 23189 / 23189
pill_crop_path missing: 0
prescription_image_path existing: 23189 / 23189
prescription_image_path missing: 0

SPLIT: val
CSV rows: 4974
Labels: 89
pill_crop_path existing: 4974 / 4974
pill_crop_path missing: 0
prescription_image_path existing: 4974 / 4974
prescription_image_path missing: 0

SPLIT: test
CSV rows: 4550
Labels: 82
pill_crop_path existing: 4550 / 4550
pill_crop_path missing: 0
prescription_image_path existing: 4550 / 4550
prescription_image_path missing: 0


In [ ]:
import os

M20_GRAPH_DIR = "/content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts"
os.makedirs(M20_GRAPH_DIR, exist_ok=True)

print("M20 prune10 graph output dir:", M20_GRAPH_DIR)

M20 prune10 graph output dir: /content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python build_m20_pruned_graph_embeddings.py \
  --base_graph_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts \
  --prune_ratio 0.10 \
  --embedding_dim 64 \
  2>&1 | tee /content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts/build_log.txt

/content/vaipe-contextual-pill-recognition
=== M20 PRUNED GRAPH EMBEDDING BUILDER ===
Base graph dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts
Prune ratio: 0.1
Embedding dim: 64
Loaded PPMI: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_ppmi.npy
PPMI shape: (108, 108)
Num labels: 108

Pruning stats:
total_positive_offdiag_edges: 926
removed_edges: 96
remaining_positive_offdiag_edges: 830
threshold: 0.18735934793949127
prune_ratio: 0.1
keep_diagonal: True

Pruned PPMI shape: (108, 108)
Embeddings shape: (108, 64)

Saved artifacts:
/content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts/build_log.txt
/content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts/graph_edges_pruned.csv
/content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts/graph_embeddings.npy
/content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_arti

In [ ]:
import os

M20_TRAIN_DIR = "/content/drive/MyDrive/model/M20_graph_pruning/train_prune10"
os.makedirs(M20_TRAIN_DIR, exist_ok=True)

print("M20 prune10 train output dir:", M20_TRAIN_DIR)

M20 prune10 train output dir: /content/drive/MyDrive/model/M20_graph_pruning/train_prune10


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m19_arch_pika_v1.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M20_graph_pruning/train_prune10 \
  --best_name M20_prune10_best.pth \
  --last_name M20_prune10_last.pth \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-4 \
  --class_weight_exponent 0.35 \
  --label_smoothing 0.02 \
  --pseudo_loss_weight 0.20 \
  --link_loss_weight 0.05 \
  --dropout_p 0.50 \
  --context_dropout_p 0.20 \
  --freeze_backbone_epochs 1 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M20_graph_pruning/train_prune10/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M19 ARCHITECTURE PIKA V1 TRAINING ===
Using device: cuda
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M20_graph_pruning/prune10_graph_artifacts
Output dir: /content/drive/MyDrive/model/M20_graph_pruning/train_prune10
Num classes: 108
Graph embeddings shape: (108, 64)
Train pill images existing: 23189 / 23189
Train prescription images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Val prescription images existing: 4974 / 4974
Train rows: 23189
Val rows: 4974
Train labels: 108
Val labels: 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64(35), n

Ý tưởng M21:

Giữ hướng PIKA/context/graph của M19
Nhưng thay kiến trúc mạnh hơn:
- Pill encoder mạnh hơn: convnext_tiny.in12k_ft_in1k
- Projection lớn hơn: common_dim = 128
- Fusion giàu hơn M19
- Multi-sample dropout classifier
- Context attention mạnh hơn
- Context length tăng từ 5 lên 8
- Có pseudo loss + graph alignment loss

Mục tiêu: vượt M19 validation 0.4907 trước, sau đó mới test.

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m21_strong_visual_pika.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 276, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 276 (delta 93), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (276/276), 2.61 MiB | 11.79 MiB/s, done.
Resolving deltas: 100% (148/148), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 28K May  6 12:16 train_m21_strong_visual_pika.py


In [ ]:
import os

M21_DIR = "/content/drive/MyDrive/model/M21_strong_visual_pika/train_run"
os.makedirs(M21_DIR, exist_ok=True)

print("M21 output dir:", M21_DIR)

M21 output dir: /content/drive/MyDrive/model/M21_strong_visual_pika/train_run


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m21_strong_visual_pika.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M21_strong_visual_pika/train_run \
  --epochs 15 \
  --batch_size 12 \
  --lr 8e-5 \
  --class_weight_exponent 0.35 \
  --label_smoothing 0.02 \
  --pseudo_loss_weight 0.20 \
  --link_loss_weight 0.05 \
  --dropout_p 0.50 \
  --context_dropout_p 0.25 \
  --max_context_len 8 \
  --common_dim 128 \
  --hidden_dim 384 \
  --freeze_backbone_epochs 1 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M21_strong_visual_pika/train_run/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M21 STRONG VISUAL PIKA TRAINING ===
Using device: cuda
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M21_strong_visual_pika/train_run
Num classes: 108
Graph embeddings shape: (108, 64)
Train pill images existing: 23189 / 23189
Train prescription images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Val prescription images existing: 4974 / 4974
Train rows: 23189
Val rows: 4974
Train labels: 108
Val labels: 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64(35), np.int64(4

In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python evaluate_m21_strong_visual_pika.py \
  --checkpoint /content/drive/MyDrive/model/M21_strong_visual_pika/train_run/M21_strong_visual_pika_best.pth \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M21_strong_visual_pika/test_eval \
  --batch_size 24 \
  2>&1 | tee /content/drive/MyDrive/model/M21_strong_visual_pika/test_eval/test_log.txt

/content/vaipe-contextual-pill-recognition
=== EVALUATE M21 STRONG VISUAL PIKA ===
Using device: cuda
Checkpoint: /content/drive/MyDrive/model/M21_strong_visual_pika/train_run/M21_strong_visual_pika_best.pth
Test CSV  : /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M21_strong_visual_pika/test_eval
Using graph embeddings: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
Num classes: 108
Max context len: 8
Graph embeddings shape: (108, 64)

Raw test rows: 4550
Raw test labels: 82
Raw test columns: ['pill_crop_path', 'pill_label', 'pill_json', 'prescription_json', 'prescription_image_path', 'context_labels', 'prescription_key', 'split', 'source_split', 'clean_split']
Test pill images existing: 4550 / 4550
Test prescription images existing: 4550 / 4550

Test rows after image check: 4550
Test labels pre

Chuyển sang hướng lớn:
1. Diagnose split + hard classes.
2. Nếu cần, tạo split v3/cross-val.
3. Train crop-only visual classifier thật mạnh.
4. Đưa visual backbone đã học vào PIKA.
5. Fine-tune PIKA.
6. Cuối cùng mới TTA/ensemble.

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh diagnose_split_and_errors.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 282 (delta 97), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (282/282), 2.62 MiB | 6.65 MiB/s, done.
Resolving deltas: 100% (152/152), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 23K May  7 06:54 diagnose_split_and_errors.py


In [ ]:
import os

DIAG_DIR = "/content/drive/MyDrive/model/diagnosis_split_errors_v1"
os.makedirs(DIAG_DIR, exist_ok=True)

print("Diagnosis output dir:", DIAG_DIR)

Diagnosis output dir: /content/drive/MyDrive/model/diagnosis_split_errors_v1


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python diagnose_split_and_errors.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --m19_predictions /content/drive/MyDrive/model/M19_arch_pika_v1/test_eval/m19_test_predictions.csv \
  --m21_predictions /content/drive/MyDrive/model/M21_strong_visual_pika/test_eval/m21_test_predictions.csv \
  --output_dir /content/drive/MyDrive/model/diagnosis_split_errors_v1 \
  2>&1 | tee /content/drive/MyDrive/model/diagnosis_split_errors_v1/diagnosis_log.txt

/content/vaipe-contextual-pill-recognition
=== DIAGNOSE SPLIT AND ERRORS ===
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
M19 predictions: /content/drive/MyDrive/model/M19_arch_pika_v1/test_eval/m19_test_predictions.csv
M21 predictions: /content/drive/MyDrive/model/M21_strong_visual_pika/test_eval/m21_test_predictions.csv
Output dir: /content/drive/MyDrive/model/diagnosis_split_errors_v1

=== SPLIT SUMMARY ===
split  rows  labels_present  min_class_count  max_class_count
train 23189             108                1             5736
  val  4974              89                1             1345
 test  4550              82                1             1296

=== LABEL COVERAGE ===
{
  "train_labels": 108,
  "val_labels": 89,
  "test_labels": 82,
  "train_missing_in_val_count": 

In [ ]:
import pandas as pd
from pathlib import Path

diag_dir = Path("/content/drive/MyDrive/model/diagnosis_split_errors_v1")

files = {
    "class_distribution": diag_dir / "class_distribution_train_val_test.csv",
    "m19_hard_classes": diag_dir / "m19_hard_classes.csv",
    "m19_confusion_pairs": diag_dir / "m19_confusion_pairs.csv",
    "m21_hard_classes": diag_dir / "m21_hard_classes.csv",
    "m21_confusion_pairs": diag_dir / "m21_confusion_pairs.csv",
    "compare_per_class": diag_dir / "compare_m19_vs_m21_per_class.csv",
    "m19_correct_m21_wrong": diag_dir / "m19_correct_m21_wrong.csv",
    "m21_correct_m19_wrong": diag_dir / "m21_correct_m19_wrong.csv",
}

for name, path in files.items():
    print(name, "=>", path.exists(), path)

class_distribution => True /content/drive/MyDrive/model/diagnosis_split_errors_v1/class_distribution_train_val_test.csv
m19_hard_classes => True /content/drive/MyDrive/model/diagnosis_split_errors_v1/m19_hard_classes.csv
m19_confusion_pairs => True /content/drive/MyDrive/model/diagnosis_split_errors_v1/m19_confusion_pairs.csv
m21_hard_classes => True /content/drive/MyDrive/model/diagnosis_split_errors_v1/m21_hard_classes.csv
m21_confusion_pairs => True /content/drive/MyDrive/model/diagnosis_split_errors_v1/m21_confusion_pairs.csv
compare_per_class => True /content/drive/MyDrive/model/diagnosis_split_errors_v1/compare_m19_vs_m21_per_class.csv
m19_correct_m21_wrong => True /content/drive/MyDrive/model/diagnosis_split_errors_v1/m19_correct_m21_wrong.csv
m21_correct_m19_wrong => True /content/drive/MyDrive/model/diagnosis_split_errors_v1/m21_correct_m19_wrong.csv


In [ ]:
import pandas as pd
from pathlib import Path

diag_dir = Path("/content/drive/MyDrive/model/diagnosis_split_errors_v1")

m19_hard = pd.read_csv(diag_dir / "m19_hard_classes.csv")
m19_conf = pd.read_csv(diag_dir / "m19_confusion_pairs.csv")
compare_cls = pd.read_csv(diag_dir / "compare_m19_vs_m21_per_class.csv")

print("\n====================")
print("1. M19 HARDEST CLASSES WITH TEST SUPPORT")
print("====================")
cols1 = [c for c in [
    "label", "support", "precision", "recall", "f1",
    "train_count", "val_count", "test_count",
    "pred_count", "error_count_est"
] if c in m19_hard.columns]

print(
    m19_hard[m19_hard["support"] > 0]
    .sort_values(["f1", "support"], ascending=[True, False])
    .head(30)[cols1]
    .to_string(index=False)
)

print("\n====================")
print("2. M19 TOP CONFUSION PAIRS")
print("====================")
cols2 = [c for c in [
    "true_norm", "pred_norm", "wrong_count",
    "true_support", "pred_support"
] if c in m19_conf.columns]

print(
    m19_conf.head(30)[cols2]
    .to_string(index=False)
)

print("\n====================")
print("3. CLASSES WHERE M21 IMPROVES OVER M19")
print("====================")
cols3 = [c for c in [
    "label", "samples", "a_correct", "b_correct",
    "a_acc", "b_acc", "b_minus_a_acc"
] if c in compare_cls.columns]

print(
    compare_cls.sort_values(["b_minus_a_acc", "samples"], ascending=[False, False])
    .head(30)[cols3]
    .to_string(index=False)
)

print("\n====================")
print("4. CLASSES WHERE M21 IS WORSE THAN M19")
print("====================")
print(
    compare_cls.sort_values(["b_minus_a_acc", "samples"], ascending=[True, False])
    .head(30)[cols3]
    .to_string(index=False)
)


1. M19 HARDEST CLASSES WITH TEST SUPPORT
 label  support  precision   recall       f1  train_count  val_count  test_count  pred_count  error_count_est
    47       16   0.000000 0.000000 0.000000           73         17          16          16             16.0
    18       10   0.000000 0.000000 0.000000           84          8          10           3             10.0
    50       10   0.000000 0.000000 0.000000           67         11          10           4             10.0
    81        8   0.000000 0.000000 0.000000           69         20           8           2              8.0
     5        7   0.000000 0.000000 0.000000           38          8           7           4              7.0
    63        6   0.000000 0.000000 0.000000           26          6           6           1              6.0
    71        4   0.000000 0.000000 0.000000           26          8           4           0              4.0
    76        4   0.000000 0.000000 0.000000           12          4          

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh evaluate_m19_m21_val_predictions.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 285, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 285 (delta 98), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (285/285), 2.62 MiB | 5.53 MiB/s, done.
Resolving deltas: 100% (153/153), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 17K May  7 07:04 evaluate_m19_m21_val_predictions.py


In [ ]:
import os

M22_VAL_DIR = "/content/drive/MyDrive/model/M22_selective_ensemble/val_predictions"
os.makedirs(M22_VAL_DIR, exist_ok=True)

print("M22 val prediction output dir:", M22_VAL_DIR)

M22 val prediction output dir: /content/drive/MyDrive/model/M22_selective_ensemble/val_predictions


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python evaluate_m19_m21_val_predictions.py \
  --eval_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --m19_checkpoint /content/drive/MyDrive/model/M19_arch_pika_v1/train_run/M19_arch_pika_v1_best.pth \
  --m21_checkpoint /content/drive/MyDrive/model/M21_strong_visual_pika/train_run/M21_strong_visual_pika_best.pth \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M22_selective_ensemble/val_predictions \
  --batch_size 24 \
  2>&1 | tee /content/drive/MyDrive/model/M22_selective_ensemble/val_predictions/val_prediction_log.txt

/content/vaipe-contextual-pill-recognition
=== EVALUATE M19 + M21 ON VALIDATION ===
Using device: cuda
Eval CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
M19 checkpoint: /content/drive/MyDrive/model/M19_arch_pika_v1/train_run/M19_arch_pika_v1_best.pth
M21 checkpoint: /content/drive/MyDrive/model/M21_strong_visual_pika/train_run/M21_strong_visual_pika_best.pth
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M22_selective_ensemble/val_predictions
Using graph embeddings: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
Using graph embeddings: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy

Graph M19 shape: (108, 64)
Graph M21 shape: (108, 64)

Raw eval rows: 4974
Raw eval labels: 89
Raw eval columns: ['pill_crop_path', 'pill_label', 'pill_json', 'prescription_json', 'prescription_image_path', 'context_labels

Điều này xác nhận: trên validation, M21 thật sự tốt hơn M19. Nhưng vì M21 test thấp hơn M19, bước M22 phải làm cẩn thận: chỉ tune ensemble trên val, sau đó apply lên test, không dùng test để chọn rule.

In [ ]:
import os

M22_TEST_DIR = "/content/drive/MyDrive/model/M22_selective_ensemble/test_predictions"
os.makedirs(M22_TEST_DIR, exist_ok=True)

print("M22 test prediction output dir:", M22_TEST_DIR)

M22 test prediction output dir: /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python evaluate_m19_m21_val_predictions.py \
  --eval_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --m19_checkpoint /content/drive/MyDrive/model/M19_arch_pika_v1/train_run/M19_arch_pika_v1_best.pth \
  --m21_checkpoint /content/drive/MyDrive/model/M21_strong_visual_pika/train_run/M21_strong_visual_pika_best.pth \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions \
  --batch_size 24 \
  2>&1 | tee /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions/test_prediction_log.txt

/content/vaipe-contextual-pill-recognition
=== EVALUATE M19 + M21 ON VALIDATION ===
Using device: cuda
Eval CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
M19 checkpoint: /content/drive/MyDrive/model/M19_arch_pika_v1/train_run/M19_arch_pika_v1_best.pth
M21 checkpoint: /content/drive/MyDrive/model/M21_strong_visual_pika/train_run/M21_strong_visual_pika_best.pth
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions
Using graph embeddings: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy
Using graph embeddings: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/graph_embeddings.npy

Graph M19 shape: (108, 64)
Graph M21 shape: (108, 64)

Raw eval rows: 4550
Raw eval labels: 82
Raw eval columns: ['pill_crop_path', 'pill_label', 'pill_json', 'prescription_json', 'prescription_image_path', 'context_labe

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh build_m22_selective_ensemble.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 288, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 288 (delta 99), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (288/288), 2.63 MiB | 7.10 MiB/s, done.
Resolving deltas: 100% (154/154), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 15K May  7 07:19 build_m22_selective_ensemble.py


In [ ]:
import os

M22_OUT = "/content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1"
os.makedirs(M22_OUT, exist_ok=True)

print("M22 output dir:", M22_OUT)

M22 output dir: /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python build_m22_selective_ensemble.py \
  --val_dir /content/drive/MyDrive/model/M22_selective_ensemble/val_predictions \
  --test_dir /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions \
  --output_dir /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1 \
  --grid_step 0.02 \
  --min_support 5 \
  --min_f1_gain 0.05 \
  2>&1 | tee /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/ensemble_log.txt

/content/vaipe-contextual-pill-recognition
/content/vaipe-contextual-pill-recognition/build_m22_selective_ensemble.py:403: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_candidates = pd.concat([global_grid, selective_grid], ignore_index=True)
=== BUILD M22 SELECTIVE ENSEMBLE ===
Val prediction dir : /content/drive/MyDrive/model/M22_selective_ensemble/val_predictions
Test prediction dir: /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions
Output dir         : /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1
Num classes: 108
Val rows: 4974
Test rows: 4550

=== BASE VAL METRICS ===
M19 Val Macro F1: 0.490893
M21 Val Macro F1: 0.515325

Trusted M21 classes: [2, 5, 6, 10, 15, 22, 28, 29, 31, 33, 36, 40, 50,

phân tích lỗi còn lại của M22

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support

m22_dir = Path("/content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1")
diag_dir = Path("/content/drive/MyDrive/model/diagnosis_split_errors_v1")

m22_pred_path = m22_dir / "test_ensemble_predictions.csv"
m19_pred_path = Path("/content/drive/MyDrive/model/M22_selective_ensemble/test_predictions/m19_val_predictions.csv")
m21_pred_path = Path("/content/drive/MyDrive/model/M22_selective_ensemble/test_predictions/m21_val_predictions.csv")
class_dist_path = diag_dir / "class_distribution_train_val_test.csv"

print("M22 pred exists:", m22_pred_path.exists(), m22_pred_path)
print("M19 pred exists:", m19_pred_path.exists(), m19_pred_path)
print("M21 pred exists:", m21_pred_path.exists(), m21_pred_path)
print("Class dist exists:", class_dist_path.exists(), class_dist_path)

m22 = pd.read_csv(m22_pred_path)
m19 = pd.read_csv(m19_pred_path)
m21 = pd.read_csv(m21_pred_path)
class_dist = pd.read_csv(class_dist_path)

print("\nM22 columns:")
print(m22.columns.tolist())

print("\nRows:")
print("M22:", len(m22))
print("M19:", len(m19))
print("M21:", len(m21))

M22 pred exists: True /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/test_ensemble_predictions.csv
M19 pred exists: True /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions/m19_val_predictions.csv
M21 pred exists: True /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions/m21_val_predictions.csv
Class dist exists: True /content/drive/MyDrive/model/diagnosis_split_errors_v1/class_distribution_train_val_test.csv

M22 columns:
['pill_crop_path', 'pill_label', 'pill_json', 'prescription_json', 'prescription_image_path', 'context_labels', 'prescription_key', 'split', 'source_split', 'clean_split', 'mapped_label', 'context_labels_mapped', 'true_mapped_label', 'pred_mapped_label', 'confidence', 'true_original_label', 'pred_original_label', 'is_correct', 'ensemble_pred_mapped_label', 'ensemble_pred_original_label', 'ensemble_confidence', 'ensemble_switched_to_m21', 'ensemble_is_correct']

Rows:
M22: 4550
M19: 4550
M21: 4550


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(y_true, y_pred, labels=None):
    acc = accuracy_score(y_true, y_pred)

    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    wp, wr, wf1, _ = precision_recall_fscore_support(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )

    return acc, p, r, f1, wf1

# Chuẩn hóa label
m22["true_label"] = m22["true_mapped_label"].astype(int)
m22["pred_label"] = m22["ensemble_pred_mapped_label"].astype(int)
m22["correct"] = m22["true_label"] == m22["pred_label"]

num_classes = 108
labels_all = list(range(num_classes))

acc, p, r, f1, wf1 = compute_metrics(
    m22["true_label"],
    m22["pred_label"],
    labels=None
)

acc_all, p_all, r_all, f1_all, wf1_all = compute_metrics(
    m22["true_label"],
    m22["pred_label"],
    labels=labels_all
)

print("=== M22 TEST METRICS RECHECK ===")
print("Accuracy:", round(acc, 6))
print("Macro Precision Present:", round(p, 6))
print("Macro Recall Present:", round(r, 6))
print("Macro F1 Present:", round(f1, 6))
print("Macro F1 All:", round(f1_all, 6))
print("Weighted F1:", round(wf1, 6))

# Per-class metrics
precision, recall, f1s, support = precision_recall_fscore_support(
    m22["true_label"],
    m22["pred_label"],
    labels=labels_all,
    zero_division=0
)

per_class = pd.DataFrame({
    "label": labels_all,
    "support": support,
    "precision": precision,
    "recall": recall,
    "f1": f1s,
})

pred_count = m22["pred_label"].value_counts().rename("pred_count")
per_class = per_class.merge(
    pred_count,
    left_on="label",
    right_index=True,
    how="left"
)
per_class["pred_count"] = per_class["pred_count"].fillna(0).astype(int)

# Add train/val/test count
class_dist["label"] = class_dist["label"].astype(int)
per_class = per_class.merge(
    class_dist[["label", "train_count", "val_count", "test_count"]],
    on="label",
    how="left"
)

per_class["error_count"] = per_class["support"] - (per_class["recall"] * per_class["support"])

print("\n====================")
print("1. M22 HARDEST CLASSES")
print("====================")
print(
    per_class[per_class["support"] > 0]
    .sort_values(["f1", "support"], ascending=[True, False])
    .head(30)
    .to_string(index=False)
)

# Confusion pairs
wrong = m22[m22["true_label"] != m22["pred_label"]].copy()

conf_pairs = (
    wrong.groupby(["true_label", "pred_label"])
    .size()
    .reset_index(name="wrong_count")
    .sort_values("wrong_count", ascending=False)
)

true_support = m22["true_label"].value_counts().rename("true_support")
pred_support = m22["pred_label"].value_counts().rename("pred_support")

conf_pairs = conf_pairs.merge(
    true_support,
    left_on="true_label",
    right_index=True,
    how="left"
)

conf_pairs = conf_pairs.merge(
    pred_support,
    left_on="pred_label",
    right_index=True,
    how="left"
)

print("\n====================")
print("2. M22 TOP CONFUSION PAIRS")
print("====================")
print(conf_pairs.head(30).to_string(index=False))

# Save files
out_dir = Path("/content/drive/MyDrive/model/M22_selective_ensemble/diagnosis_after_m22")
out_dir.mkdir(parents=True, exist_ok=True)

per_class.to_csv(out_dir / "m22_per_class_metrics.csv", index=False)
conf_pairs.to_csv(out_dir / "m22_confusion_pairs.csv", index=False)

print("\nSaved:")
print(out_dir / "m22_per_class_metrics.csv")
print(out_dir / "m22_confusion_pairs.csv")

=== M22 TEST METRICS RECHECK ===
Accuracy: 0.758022
Macro Precision Present: 0.491424
Macro Recall Present: 0.466117
Macro F1 Present: 0.46245
Macro F1 All: 0.411067
Weighted F1: 0.750668

1. M22 HARDEST CLASSES
 label  support  precision   recall       f1  pred_count  train_count  val_count  test_count  error_count
    80       16   0.000000 0.000000 0.000000           3           64         12          16         16.0
    18       10   0.000000 0.000000 0.000000           2           84          8          10         10.0
    81        8   0.000000 0.000000 0.000000           2           69         20           8          8.0
     5        7   0.000000 0.000000 0.000000           2           38          8           7          7.0
    48        7   0.000000 0.000000 0.000000           3           50         12           7          7.0
    71        4   0.000000 0.000000 0.000000           1           26          8           4          4.0
    76        4   0.000000 0.000000 0.000000  

In [ ]:
# Align M19 and M22 by pill_crop_path
m19_small = m19[[
    "pill_crop_path",
    "true_mapped_label",
    "pred_mapped_label",
    "is_correct"
]].copy()

m19_small = m19_small.rename(columns={
    "pred_mapped_label": "m19_pred",
    "is_correct": "m19_correct"
})

m22_small = m22[[
    "pill_crop_path",
    "true_mapped_label",
    "ensemble_pred_mapped_label",
    "ensemble_is_correct",
    "ensemble_switched_to_m21"
]].copy()

m22_small = m22_small.rename(columns={
    "ensemble_pred_mapped_label": "m22_pred",
    "ensemble_is_correct": "m22_correct"
})

cmp = m19_small.merge(
    m22_small,
    on=["pill_crop_path", "true_mapped_label"],
    how="inner"
)

cmp["case"] = "unknown"
cmp.loc[cmp["m19_correct"] & cmp["m22_correct"], "case"] = "both_correct"
cmp.loc[cmp["m19_correct"] & ~cmp["m22_correct"], "case"] = "m19_only_correct"
cmp.loc[~cmp["m19_correct"] & cmp["m22_correct"], "case"] = "m22_only_correct"
cmp.loc[~cmp["m19_correct"] & ~cmp["m22_correct"], "case"] = "both_wrong"

case_summary = (
    cmp["case"]
    .value_counts()
    .rename_axis("case")
    .reset_index(name="count")
)

case_summary["ratio"] = case_summary["count"] / len(cmp)

print("====================")
print("3. M19 VS M22 CASE SUMMARY")
print("====================")
print(case_summary.to_string(index=False))

per_class_cmp = (
    cmp.groupby("true_mapped_label")
    .agg(
        samples=("pill_crop_path", "count"),
        m19_correct=("m19_correct", "sum"),
        m22_correct=("m22_correct", "sum"),
        m22_switched=("ensemble_switched_to_m21", "sum")
    )
    .reset_index()
    .rename(columns={"true_mapped_label": "label"})
)

per_class_cmp["m19_acc"] = per_class_cmp["m19_correct"] / per_class_cmp["samples"]
per_class_cmp["m22_acc"] = per_class_cmp["m22_correct"] / per_class_cmp["samples"]
per_class_cmp["m22_minus_m19"] = per_class_cmp["m22_acc"] - per_class_cmp["m19_acc"]

print("\n====================")
print("4. CLASSES WHERE M22 IMPROVES OVER M19")
print("====================")
print(
    per_class_cmp
    .sort_values(["m22_minus_m19", "samples"], ascending=[False, False])
    .head(30)
    .to_string(index=False)
)

print("\n====================")
print("5. CLASSES WHERE M22 IS WORSE THAN M19")
print("====================")
print(
    per_class_cmp
    .sort_values(["m22_minus_m19", "samples"], ascending=[True, False])
    .head(30)
    .to_string(index=False)
)

out_dir = Path("/content/drive/MyDrive/model/M22_selective_ensemble/diagnosis_after_m22")
cmp.to_csv(out_dir / "m19_vs_m22_row_level.csv", index=False)
case_summary.to_csv(out_dir / "m19_vs_m22_case_summary.csv", index=False)
per_class_cmp.to_csv(out_dir / "m19_vs_m22_per_class_compare.csv", index=False)

print("\nSaved:")
print(out_dir / "m19_vs_m22_row_level.csv")
print(out_dir / "m19_vs_m22_case_summary.csv")
print(out_dir / "m19_vs_m22_per_class_compare.csv")

3. M19 VS M22 CASE SUMMARY
            case  count    ratio
    both_correct   3194 0.701978
      both_wrong    897 0.197143
m22_only_correct    255 0.056044
m19_only_correct    204 0.044835

4. CLASSES WHERE M22 IMPROVES OVER M19
 label  samples  m19_correct  m22_correct  m22_switched  m19_acc  m22_acc  m22_minus_m19
    68       16            1            7             3 0.062500 0.437500       0.375000
    21        3            0            1             2 0.000000 0.333333       0.333333
    31       16           11           16            16 0.687500 1.000000       0.312500
    50       10            0            3             1 0.000000 0.300000       0.300000
    84       30           17           25             0 0.566667 0.833333       0.266667
   100        4            2            3             3 0.500000 0.750000       0.250000
    72       14            1            4             4 0.071429 0.285714       0.214286
    63        6            0            1             3 

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m23_stage1_visual_classifier.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 291, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 291 (delta 100), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (291/291), 2.63 MiB | 4.18 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 23K May  7 07:32 train_m23_stage1_visual_classifier.py


In [ ]:
import os

M23_STAGE1_DIR = "/content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run"
os.makedirs(M23_STAGE1_DIR, exist_ok=True)

print("M23 Stage1 output dir:", M23_STAGE1_DIR)

M23 Stage1 output dir: /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run


Mốc để theo dõi

M23 Stage 1 là crop-only visual classifier, nên chưa so trực tiếp với M22 context ensemble. Nhưng nó phải đạt validation đủ tốt để đáng dùng làm backbone cho M24.

In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m23_stage1_visual_classifier.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run \
  --backbone_name convnext_tiny.in12k_ft_in1k \
  --epochs 20 \
  --batch_size 32 \
  --lr 5e-5 \
  --class_weight_exponent 0.35 \
  --focal_gamma 1.5 \
  --label_smoothing 0.02 \
  --use_weighted_sampler \
  --sampler_exponent 0.5 \
  --max_sample_weight_ratio 20.0 \
  --freeze_backbone_epochs 1 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M23 STAGE 1 VISUAL PRETRAINING ===
Using device: cuda
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run
Num classes: 108
Train pill images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Train rows: 23189
Val rows: 4974
Train labels: 108
Val labels: 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64(35), np.int64(44), np.int64(49), np.int64(53), np.int64(66), np.int64(85), np.int64(86), np.int64(94), np.int64(96), np.int64(102)]
Using W

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m23_stage1_visual_classifier_v2.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 294, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 294 (delta 101), reused 22 (delta 22), pack-reused 122 (from 1)
Receiving objects: 100% (294/294), 2.64 MiB | 5.68 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 16K May  7 08:35 train_m23_stage1_visual_classifier_v2.py


In [ ]:
import os

M23_STAGE1_V2_DIR = "/content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run_v2_ce"
os.makedirs(M23_STAGE1_V2_DIR, exist_ok=True)

print("M23 Stage1 v2 output dir:", M23_STAGE1_V2_DIR)

M23 Stage1 v2 output dir: /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run_v2_ce


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m23_stage1_visual_classifier_v2.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --output_dir /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run_v2_ce \
  --backbone_name convnext_tiny.in12k_ft_in1k \
  --epochs 20 \
  --batch_size 32 \
  --lr 3e-5 \
  --class_weight_exponent 0.20 \
  --label_smoothing 0.05 \
  --dropout_p 0.55 \
  --freeze_backbone_epochs 1 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run_v2_ce/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M23 STAGE 1 VISUAL PRETRAINING V2 ===
Using device: cuda
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
Output dir: /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run_v2_ce
Num classes: 108
Train pill images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Train rows: 23189
Val rows: 4974
Train labels: 108
Val labels: 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64(35), np.int64(44), np.int64(49), np.int64(53), np.int64(66), np.int64(85), np.int64(86), np.int64(94), np.int64(96), np.int64(102)

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh evaluate_m23_stage1_visual_classifier.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 297 (delta 23), reused 22 (delta 22), pack-reused 271 (from 2)
Receiving objects: 100% (297/297), 2.64 MiB | 4.33 MiB/s, done.
Resolving deltas: 100% (158/158), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 9.8K May  7 09:41 evaluate_m23_stage1_visual_classifier.py


In [ ]:
import os

M23_V2_TEST_DIR = "/content/drive/MyDrive/model/M23_stage1_visual_pretraining/test_eval_v2_ce"
os.makedirs(M23_V2_TEST_DIR, exist_ok=True)

print("M23 v2 test output dir:", M23_V2_TEST_DIR)

M23 v2 test output dir: /content/drive/MyDrive/model/M23_stage1_visual_pretraining/test_eval_v2_ce


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python evaluate_m23_stage1_visual_classifier.py \
  --checkpoint /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run_v2_ce/M23_stage1_v2_visual_best.pth \
  --eval_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --output_dir /content/drive/MyDrive/model/M23_stage1_visual_pretraining/test_eval_v2_ce \
  --batch_size 32 \
  2>&1 | tee /content/drive/MyDrive/model/M23_stage1_visual_pretraining/test_eval_v2_ce/test_log.txt

/content/vaipe-contextual-pill-recognition
=== EVALUATE M23 STAGE1 VISUAL CLASSIFIER ===
Using device: cuda
Checkpoint: /content/drive/MyDrive/model/M23_stage1_visual_pretraining/train_run_v2_ce/M23_stage1_v2_visual_best.pth
Eval CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Output dir: /content/drive/MyDrive/model/M23_stage1_visual_pretraining/test_eval_v2_ce
Model type: M23VisualClassifier
Stage: M23_stage1_visual_pretraining_v2
Epoch: 14
Checkpoint Val Macro F1 Present: 0.43270142577402326
Num classes: 108
Backbone: convnext_tiny.in12k_ft_in1k
Hidden dim: 512
Dropout: 0.55
Image size: 224
Eval pill images existing: 4550 / 4550

Eval rows: 4550
Eval labels: 82
Mapped label min: 1
Mapped label max: 107
Evaluating M23 visual: 100%|██████████| 143/143 [00:34<00:00,  4.20it/s]

=== M23 VISUAL EVAL RESULTS ===
Accuracy                 : 0.660879
Macro Precision Present  : 0.415024
Macro Recall Present     : 0.382786
Macro F1 Present Classes : 0.379619
M

Dọn repo và benchmark:
- README tách old validation result và clean test result.
- Tạo EXPERIMENTS_M17_M23.md.
- Tạo audit_pika_paper_protocol.py.

Tạo paper-compatible benchmark:
- Nếu mục tiêu báo cáo là vượt paper, phải có track này.

M24 metric visual training:
- CE + SupCon + ArcFace/CosFace.
- hard pair sampler.
- backbone DINOv2/ConvNeXtV2/EVA/ConvNeXt base.

Evaluate M24 trên val/test:
- Nếu M24 visual-only không vượt M23 rõ rệt, chỉnh metric learning.
- Nếu M24 cứu hard classes, đưa vào ensemble.

M26 context PIKA với M24 pretrained visual:
- Kế thừa M19 architecture.
- Thêm hard-pair auxiliary loss.

M27 stacked ensemble:
- M19 + M21 + M24 + M26.
- Tune bằng validation only.
- Apply test once.

Nếu cần full pipeline:
- train detector hiện đại.
- đánh giá end-to-end.

In [ ]:
import os

AUDIT_DIR = "/content/drive/MyDrive/model/audit_pika_protocol_v1"
os.makedirs(AUDIT_DIR, exist_ok=True)

print("Audit output dir:", AUDIT_DIR)

Audit output dir: /content/drive/MyDrive/model/audit_pika_protocol_v1


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python audit_pika_paper_protocol.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --m19_predictions /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions/m19_val_predictions.csv \
  --m22_predictions /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/test_ensemble_predictions.csv \
  --output_dir /content/drive/MyDrive/model/audit_pika_protocol_v1 \
  2>&1 | tee /content/drive/MyDrive/model/audit_pika_protocol_v1/audit_log.txt

/content/vaipe-contextual-pill-recognition
=== AUDIT PIKA PAPER PROTOCOL / CLEAN SPLIT ===
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Output dir: /content/drive/MyDrive/model/audit_pika_protocol_v1

=== SPLIT SUMMARY ===
split  rows  groups  labels_present  min_class_count  max_class_count  median_class_count  mean_class_count
train 23189     597             108                1             5736                75.5        214.712963
  val  4974     219              89                1             1345                20.0         55.887640
 test  4550     357              82                1             1296                18.5         55.487805

=== PATH CHECK ===
split  pill_crop_path_existing  pill_crop_path_missing  pill_crop_path_total  prescription_image_path_existin

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh evaluate_candidate_benchmarks.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 303, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 303 (delta 25), reused 22 (delta 22), pack-reused 271 (from 2)
Receiving objects: 100% (303/303), 2.64 MiB | 4.84 MiB/s, done.
Resolving deltas: 100% (160/160), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 6.3K May  7 11:12 evaluate_candidate_benchmarks.py


In [ ]:
import os

CANDIDATE_EVAL_DIR = "/content/drive/MyDrive/model/audit_pika_protocol_v1/candidate_eval"
os.makedirs(CANDIDATE_EVAL_DIR, exist_ok=True)

print("Candidate eval output dir:", CANDIDATE_EVAL_DIR)

Candidate eval output dir: /content/drive/MyDrive/model/audit_pika_protocol_v1/candidate_eval


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python evaluate_candidate_benchmarks.py \
  --candidates_csv /content/drive/MyDrive/model/audit_pika_protocol_v1/audit_candidate_benchmarks.csv \
  --m19_predictions /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions/m19_val_predictions.csv \
  --m22_predictions /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/test_ensemble_predictions.csv \
  --output_dir /content/drive/MyDrive/model/audit_pika_protocol_v1/candidate_eval \
  2>&1 | tee /content/drive/MyDrive/model/audit_pika_protocol_v1/candidate_eval/candidate_eval_log.txt

/content/vaipe-contextual-pill-recognition
=== EVALUATE CANDIDATE BENCHMARKS ===
Candidate file: /content/drive/MyDrive/model/audit_pika_protocol_v1/audit_candidate_benchmarks.csv
M19 predictions: /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions/m19_val_predictions.csv
M22 predictions: /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/test_ensemble_predictions.csv
Output dir: /content/drive/MyDrive/model/audit_pika_protocol_v1/candidate_eval

=== CANDIDATE BENCHMARK RESULTS ===
model                          candidate  candidate_num_labels  eval_rows  accuracy  macro_f1_present  macro_f1_all_candidate  weighted_f1
  M19                   all_train_labels                   108       4550  0.746813          0.443011                0.397890     0.744192
  M22                   all_train_labels                   108       4550  0.758022          0.462450                0.411067     0.750668
  M19        labels_present_in_train_val                    89


Tạo hard-pair metadata từ M22 confusion.


Train M24 metric visual model:
- backbone mạnh hơn
- CE + SupCon / ArcFace
- hard-pair sampler


Evaluate M24 visual-only:
- tổng metric
- hard classes
- M24 đúng/M22 sai


Nếu M24 cứu được hard classes:
- đưa M24 vào ensemble hoặc PIKA context model.


Nếu M24 không cứu hard classes:
- đổi backbone lên DINOv2 / EVA / ConvNeXt Base.

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh build_m24_hard_pair_metadata.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 306, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 306 (delta 26), reused 22 (delta 22), pack-reused 271 (from 2)
Receiving objects: 100% (306/306), 2.65 MiB | 6.51 MiB/s, done.
Resolving deltas: 100% (161/161), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 17K May  7 11:20 build_m24_hard_pair_metadata.py


In [ ]:
import os

M24_META_DIR = "/content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata"
os.makedirs(M24_META_DIR, exist_ok=True)

print("M24 metadata output dir:", M24_META_DIR)

M24 metadata output dir: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python build_m24_hard_pair_metadata.py \
  --train_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv \
  --val_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv \
  --test_csv /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv \
  --graph_artifacts_dir /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts \
  --m22_confusion_pairs /content/drive/MyDrive/model/M22_selective_ensemble/diagnosis_after_m22/m22_confusion_pairs.csv \
  --m22_per_class_metrics /content/drive/MyDrive/model/M22_selective_ensemble/diagnosis_after_m22/m22_per_class_metrics.csv \
  --output_dir /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata \
  --min_wrong_count 5 \
  --top_k_per_true 3 \
  --make_symmetric \
  --f1_threshold 0.45 \
  --recall_threshold 0.35 \
  --min_support 3 \
  --min_error_count 7 \
  --hard_weight_boost 1.5 \
  2>&1 | tee /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/build_log.txt

/content/vaipe-contextual-pill-recognition
=== BUILD M24 HARD PAIR METADATA ===
Train CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/train_clean.csv
Val CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/val_clean.csv
Test CSV: /content/drive/MyDrive/vaipe_splits/clean_paper_like_split_v2/test_clean.csv
Graph artifacts dir: /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts
M22 confusion pairs: /content/drive/MyDrive/model/M22_selective_ensemble/diagnosis_after_m22/m22_confusion_pairs.csv
M22 per-class metrics: /content/drive/MyDrive/model/M22_selective_ensemble/diagnosis_after_m22/m22_per_class_metrics.csv
Output dir: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata

=== HARD PAIR EDGES TOP 30 ===
 true_label  pred_label  wrong_count  true_support  pred_support
         89           0           25           180            26
         59          73           22            85            62
         36          51       

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m24_metric_visual.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 309, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 309 (delta 27), reused 22 (delta 22), pack-reused 271 (from 2)
Receiving objects: 100% (309/309), 2.66 MiB | 5.72 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 32K May  7 11:30 train_m24_metric_visual.py


In [ ]:
import os

M24_RUN_DIR = "/content/drive/MyDrive/model/M24_metric_learning/train_run_v1"
os.makedirs(M24_RUN_DIR, exist_ok=True)

print("M24 run output dir:", M24_RUN_DIR)

M24 run output dir: /content/drive/MyDrive/model/M24_metric_learning/train_run_v1


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m24_metric_visual.py \
  --train_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv \
  --val_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv \
  --test_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv \
  --hard_negative_map /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_hard_negative_map.json \
  --idx_to_label /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/idx_to_label.json \
  --output_dir /content/drive/MyDrive/model/M24_metric_learning/train_run_v1 \
  --backbone_name convnext_tiny.in12k_ft_in1k \
  --epochs 20 \
  --classes_per_batch 16 \
  --samples_per_class 2 \
  --eval_batch_size 32 \
  --lr 3e-5 \
  --class_weight_exponent 0.15 \
  --hard_weight_max 1.8 \
  --label_smoothing 0.03 \
  --cosface_margin 0.15 \
  --supcon_weight 0.10 \
  --hardpair_weight 0.05 \
  --freeze_backbone_epochs 1 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M24_metric_learning/train_run_v1/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M24 METRIC VISUAL TRAINING ===
Using device: cuda
Train metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv
Val metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv
Test metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv
Hard negative map: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_hard_negative_map.json
Output dir: /content/drive/MyDrive/model/M24_metric_learning/train_run_v1
Train pill images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Num classes: 108
Train rows: 23189
Val rows: 4974
Train labels: 108
Val labels: 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64(35), np.int64(44), np.int64(49), np.int64(53), np.i

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m24_metric_visual_v2.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 312 (delta 29), reused 22 (delta 22), pack-reused 271 (from 2)
Receiving objects: 100% (312/312), 2.66 MiB | 7.63 MiB/s, done.
Resolving deltas: 100% (164/164), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 28K May  7 12:01 train_m24_metric_visual_v2.py


In [ ]:
import os

M24_V2_DIR = "/content/drive/MyDrive/model/M24_metric_learning/train_run_v2_stable"
os.makedirs(M24_V2_DIR, exist_ok=True)

print("M24 v2 output dir:", M24_V2_DIR)

M24 v2 output dir: /content/drive/MyDrive/model/M24_metric_learning/train_run_v2_stable


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m24_metric_visual_v2.py \
  --train_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv \
  --val_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv \
  --test_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv \
  --idx_to_label /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/idx_to_label.json \
  --output_dir /content/drive/MyDrive/model/M24_metric_learning/train_run_v2_stable \
  --backbone_name convnext_tiny.in12k_ft_in1k \
  --epochs 20 \
  --classes_per_batch 16 \
  --samples_per_class 2 \
  --eval_batch_size 32 \
  --lr 3e-5 \
  --class_weight_exponent 0.12 \
  --hard_weight_max 1.3 \
  --label_smoothing 0.05 \
  --dropout_p 0.45 \
  --supcon_weight 0.05 \
  --freeze_backbone_epochs 1 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M24_metric_learning/train_run_v2_stable/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M24 METRIC VISUAL V2 TRAINING ===
Using device: cuda
Train metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv
Val metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv
Test metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv
Output dir: /content/drive/MyDrive/model/M24_metric_learning/train_run_v2_stable
Train pill images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Num classes: 108
Train rows: 23189
Val rows: 4974
Train labels: 108
Val labels: 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64(35), np.int64(44), np.int64(49), np.int64(53), np.int64(66), np.int64(85), np.int64(86), np.int64(94), np.int64(96), np.int64(102)]
Created M24MetricVisual

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh train_m25_strong_visual_ce_natural.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 317, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 317 (delta 32), reused 22 (delta 22), pack-reused 271 (from 2)
Receiving objects: 100% (317/317), 2.66 MiB | 7.43 MiB/s, done.
Resolving deltas: 100% (167/167), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 27K May  7 12:22 train_m25_strong_visual_ce_natural.py


In [ ]:
import os

M25_RUN_DIR = "/content/drive/MyDrive/model/M25_strong_visual_ce_natural/train_run_v1"
os.makedirs(M25_RUN_DIR, exist_ok=True)

print("M25 output dir:", M25_RUN_DIR)

M25 output dir: /content/drive/MyDrive/model/M25_strong_visual_ce_natural/train_run_v1


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m25_strong_visual_ce_natural.py \
  --train_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv \
  --val_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv \
  --test_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv \
  --idx_to_label /content/drive/MyDrive/model/M17_faithful_pika/graph_artifacts/idx_to_label.json \
  --output_dir /content/drive/MyDrive/model/M25_strong_visual_ce_natural/train_run_v1 \
  --backbone_name tf_efficientnetv2_s.in21k_ft_in1k \
  --epochs 20 \
  --batch_size 24 \
  --eval_batch_size 32 \
  --backbone_lr 1e-5 \
  --head_lr 5e-5 \
  --class_weight_exponent 0.05 \
  --hard_weight_max 1.05 \
  --label_smoothing 0.03 \
  --dropout_p 0.45 \
  --freeze_backbone_epochs 1 \
  --use_ema \
  --ema_decay 0.999 \
  --patience 5 \
  2>&1 | tee /content/drive/MyDrive/model/M25_strong_visual_ce_natural/train_run_v1/train_log.txt

/content/vaipe-contextual-pill-recognition
=== M25 STRONG VISUAL CE NATURAL TRAINING ===
Using device: cuda
Train metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv
Val metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv
Test metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv
Output dir: /content/drive/MyDrive/model/M25_strong_visual_ce_natural/train_run_v1
Train pill images existing: 23189 / 23189
Val pill images existing: 4974 / 4974
Num classes: 108
Train rows: 23189
Val rows: 4974
Train labels: 108
Val labels: 89
Labels in train but missing in val: 19
[np.int64(0), np.int64(3), np.int64(4), np.int64(9), np.int64(13), np.int64(16), np.int64(20), np.int64(25), np.int64(32), np.int64(35), np.int64(44), np.int64(49), np.int64(53), np.int64(66), np.int64(85), np.int64(86), np.int64(94), np.int64(96), np.int64(102)]
Created M25St

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh build_m26_calibrated_context_ensemble.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 320, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 320 (delta 33), reused 22 (delta 22), pack-reused 271 (from 2)
Receiving objects: 100% (320/320), 2.67 MiB | 7.49 MiB/s, done.
Resolving deltas: 100% (168/168), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 16K May  7 13:51 build_m26_calibrated_context_ensemble.py


In [ ]:
import os

M26_DIR = "/content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1"
os.makedirs(M26_DIR, exist_ok=True)

print("M26 output dir:", M26_DIR)

M26 output dir: /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python build_m26_calibrated_context_ensemble.py \
  --val_m19_m21_dir /content/drive/MyDrive/model/M22_selective_ensemble/val_predictions \
  --test_m19_m21_dir /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions \
  --m22_dir /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1 \
  --train_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv \
  --output_dir /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1 \
  --weight_step 0.05 \
  --logit_adjust_alphas="-0.50,-0.25,0.00,0.10,0.20,0.30,0.40,0.50,0.75,1.00" \
  --temperatures="0.75,1.00,1.25,1.50,2.00" \
  2>&1 | tee /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1/m26_log.txt

/content/vaipe-contextual-pill-recognition
=== M26 CALIBRATED CONTEXT ENSEMBLE ===
Val M19/M21 dir : /content/drive/MyDrive/model/M22_selective_ensemble/val_predictions
Test M19/M21 dir: /content/drive/MyDrive/model/M22_selective_ensemble/test_predictions
M22 dir         : /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1
Train metadata  : /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv
Output dir      : /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1
Num classes: 108
Val rows: 4974
Test rows: 4550
Train prior min/max: 8.584796325707173e-05 0.24625488260291026

=== BASELINE METRICS ===
split model  accuracy  macro_f1_present  macro_f1_all  weighted_f1
  val   M19  0.749698          0.490893      0.445440     0.748022
  val   M21  0.756534          0.515325      0.472381     0.750121
  val   M22  0.774829          0.550088      0.488967     0.767312
 test   M19  0.746813          0.443011      0.397890

In [ ]:
import pandas as pd
from pathlib import Path

m22_path = Path("/content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/test_ensemble_predictions.csv")
m26_path = Path("/content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1/test_m26_predictions.csv")

print("M22 exists:", m22_path.exists(), m22_path)
print("M26 exists:", m26_path.exists(), m26_path)

m22 = pd.read_csv(m22_path)
m26 = pd.read_csv(m26_path)

m22_small = m22[[
    "pill_crop_path",
    "true_mapped_label",
    "ensemble_pred_mapped_label",
    "ensemble_is_correct"
]].copy()

m22_small = m22_small.rename(columns={
    "ensemble_pred_mapped_label": "m22_pred",
    "ensemble_is_correct": "m22_correct"
})

m26_small = m26[[
    "pill_crop_path",
    "true_mapped_label",
    "m26_pred_mapped_label",
    "m26_is_correct"
]].copy()

m26_small = m26_small.rename(columns={
    "m26_pred_mapped_label": "m26_pred",
    "m26_is_correct": "m26_correct"
})

cmp = m22_small.merge(
    m26_small,
    on=["pill_crop_path", "true_mapped_label"],
    how="inner"
)

cmp["case"] = "unknown"
cmp.loc[cmp["m22_correct"] & cmp["m26_correct"], "case"] = "both_correct"
cmp.loc[cmp["m22_correct"] & ~cmp["m26_correct"], "case"] = "m22_only_correct"
cmp.loc[~cmp["m22_correct"] & cmp["m26_correct"], "case"] = "m26_only_correct"
cmp.loc[~cmp["m22_correct"] & ~cmp["m26_correct"], "case"] = "both_wrong"

case_summary = (
    cmp["case"]
    .value_counts()
    .rename_axis("case")
    .reset_index(name="count")
)

case_summary["ratio"] = case_summary["count"] / len(cmp)

print("\n====================")
print("1. M22 VS M26 CASE SUMMARY")
print("====================")
print(case_summary.to_string(index=False))

per_class_cmp = (
    cmp.groupby("true_mapped_label")
    .agg(
        samples=("pill_crop_path", "count"),
        m22_correct=("m22_correct", "sum"),
        m26_correct=("m26_correct", "sum")
    )
    .reset_index()
    .rename(columns={"true_mapped_label": "label"})
)

per_class_cmp["m22_acc"] = per_class_cmp["m22_correct"] / per_class_cmp["samples"]
per_class_cmp["m26_acc"] = per_class_cmp["m26_correct"] / per_class_cmp["samples"]
per_class_cmp["m26_minus_m22"] = per_class_cmp["m26_acc"] - per_class_cmp["m22_acc"]

print("\n====================")
print("2. CLASSES WHERE M26 IMPROVES OVER M22")
print("====================")
print(
    per_class_cmp
    .sort_values(["m26_minus_m22", "samples"], ascending=[False, False])
    .head(30)
    .to_string(index=False)
)

print("\n====================")
print("3. CLASSES WHERE M26 IS WORSE THAN M22")
print("====================")
print(
    per_class_cmp
    .sort_values(["m26_minus_m22", "samples"], ascending=[True, False])
    .head(30)
    .to_string(index=False)
)

# Confusion pairs của M26
wrong = m26[m26["true_mapped_label"] != m26["m26_pred_mapped_label"]].copy()

conf_pairs = (
    wrong.groupby(["true_mapped_label", "m26_pred_mapped_label"])
    .size()
    .reset_index(name="wrong_count")
    .sort_values("wrong_count", ascending=False)
)

true_support = m26["true_mapped_label"].value_counts().rename("true_support")
pred_support = m26["m26_pred_mapped_label"].value_counts().rename("pred_support")

conf_pairs = conf_pairs.merge(
    true_support,
    left_on="true_mapped_label",
    right_index=True,
    how="left"
)

conf_pairs = conf_pairs.merge(
    pred_support,
    left_on="m26_pred_mapped_label",
    right_index=True,
    how="left"
)

print("\n====================")
print("4. M26 TOP CONFUSION PAIRS")
print("====================")
print(conf_pairs.head(30).to_string(index=False))

out_dir = Path("/content/drive/MyDrive/model/M26_calibrated_context_ensemble/diagnosis_vs_m22")
out_dir.mkdir(parents=True, exist_ok=True)

cmp.to_csv(out_dir / "m22_vs_m26_row_level.csv", index=False)
case_summary.to_csv(out_dir / "m22_vs_m26_case_summary.csv", index=False)
per_class_cmp.to_csv(out_dir / "m22_vs_m26_per_class_compare.csv", index=False)
conf_pairs.to_csv(out_dir / "m26_confusion_pairs.csv", index=False)

print("\nSaved:")
print(out_dir / "m22_vs_m26_row_level.csv")
print(out_dir / "m22_vs_m26_case_summary.csv")
print(out_dir / "m22_vs_m26_per_class_compare.csv")
print(out_dir / "m26_confusion_pairs.csv")

M22 exists: True /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/test_ensemble_predictions.csv
M26 exists: True /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1/test_m26_predictions.csv

1. M22 VS M26 CASE SUMMARY
            case  count    ratio
    both_correct   3418 0.751209
      both_wrong   1001 0.220000
m26_only_correct    100 0.021978
m22_only_correct     31 0.006813

2. CLASSES WHERE M26 IMPROVES OVER M22
 label  samples  m22_correct  m26_correct  m22_acc  m26_acc  m26_minus_m22
    75       10            2            4 0.200000 0.400000       0.200000
    63        6            1            2 0.166667 0.333333       0.166667
    67       15           11           13 0.733333 0.866667       0.133333
    62        9            8            9 0.888889 1.000000       0.111111
    12        9            7            8 0.777778 0.888889       0.111111
    70       37           12           15 0.324324 0.405405       0.081081
    41       13  

In [ ]:
%cd /content
!rm -rf vaipe-contextual-pill-recognition
!git clone https://github.com/2274802010922/vaipe-contextual-pill-recognition.git
%cd /content/vaipe-contextual-pill-recognition

!ls -lh build_m27_classwise_fallback_ensemble.py

/content
Cloning into 'vaipe-contextual-pill-recognition'...
remote: Enumerating objects: 323, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 323 (delta 34), reused 22 (delta 22), pack-reused 271 (from 2)
Receiving objects: 100% (323/323), 2.67 MiB | 7.28 MiB/s, done.
Resolving deltas: 100% (169/169), done.
/content/vaipe-contextual-pill-recognition
-rw-r--r-- 1 root root 17K May  7 14:13 build_m27_classwise_fallback_ensemble.py


In [ ]:
import os

M27_DIR = "/content/drive/MyDrive/model/M27_classwise_fallback_ensemble/run_v1"
os.makedirs(M27_DIR, exist_ok=True)

print("M27 output dir:", M27_DIR)

M27 output dir: /content/drive/MyDrive/model/M27_classwise_fallback_ensemble/run_v1


In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python build_m27_classwise_fallback_ensemble.py \
  --m22_dir /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1 \
  --m26_dir /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1 \
  --output_dir /content/drive/MyDrive/model/M27_classwise_fallback_ensemble/run_v1 \
  2>&1 | tee /content/drive/MyDrive/model/M27_classwise_fallback_ensemble/run_v1/m27_log.txt

/content/vaipe-contextual-pill-recognition
=== M27 CLASS-WISE FALLBACK ENSEMBLE ===
M22 dir: /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1
M26 dir: /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1
Output dir: /content/drive/MyDrive/model/M27_classwise_fallback_ensemble/run_v1
Num classes: 108
Val rows: 4974
Test rows: 4550

=== BASE METRICS ===
split model  accuracy  macro_f1_present  macro_f1_all  weighted_f1
  val   M22  0.774829          0.550088      0.488967     0.767312
  val   M26  0.782871          0.563060      0.495284     0.771756
 test   M22  0.758022          0.462450      0.411067     0.750668
 test   M26  0.773187          0.500298      0.421548     0.762100

=== LOW M26 PREDICTED-CLASS RELIABILITY TOP 30 ===
 pred_class  m22_pred_support  m22_pred_precision  m26_pred_support  m26_pred_precision
         44                14            0.000000                13            0.000000
         85                12            0.000000

In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python analyze_m28_m22_m26_complementarity.py \
  --m22_dir /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1 \
  --m26_dir /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1 \
  --output_dir /content/drive/MyDrive/model/M28_error_driven_soft_ensemble/run_v1 \
  --num_classes 108 \
  --true_col true_mapped_label

/content/vaipe-contextual-pill-recognition
=== M28 ERROR-DRIVEN CALIBRATED SOFT ENSEMBLE ===
M22 dir    : /content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1
M26 dir    : /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1
Output dir : /content/drive/MyDrive/model/M28_error_driven_soft_ensemble/run_v1
Num classes: 108

=== INPUT FILES ===
{
  "val": {
    "m22_csv": "/content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/val_ensemble_predictions.csv",
    "m26_csv": "/content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1/val_m26_predictions.csv",
    "m22_probs": "/content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/val_ensemble_probs.npy",
    "m26_probs": "/content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1/val_m26_probs.npy"
  },
  "test": {
    "m22_csv": "/content/drive/MyDrive/model/M22_selective_ensemble/ensemble_run_v1/test_ensemble_predictions.csv",
    "m26_csv": "/content/drive/MyDrive/m

In [ ]:
%cd /content/vaipe-contextual-pill-recognition

!python run_m29_classwise_logit_bias_calibration.py \
  --m26_dir /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1 \
  --output_dir /content/drive/MyDrive/model/M29_classwise_logit_bias_calibration/run_v1 \
  --num_classes 108 \
  --true_col true_mapped_label

/content/vaipe-contextual-pill-recognition
=== M29 CLASS-WISE LOGIT BIAS CALIBRATION ===
M26 dir    : /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1
Output dir : /content/drive/MyDrive/model/M29_classwise_logit_bias_calibration/run_v1
Num classes: 108
Selection metric: macro_f1_present
Max accuracy drop allowed before penalty: 0.005

=== INPUT FILES ===
{
  "val": {
    "csv": "/content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1/val_m26_predictions.csv",
    "probs": "/content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1/val_m26_probs.npy",
    "true_col": "true_mapped_label"
  },
  "test": {
    "csv": "/content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1/test_m26_predictions.csv",
    "probs": "/content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1/test_m26_probs.npy",
    "true_col": "true_mapped_label"
  }
}

=== BASE M26 METRICS ===
split model  accuracy  macro_f1_present  macro_f1_all  weighted_f1
  va

In [7]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m30_hard_error_finetune.py \
  --train_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv \
  --val_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv \
  --test_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv \
  --m26_dir /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1 \
  --output_dir /content/drive/MyDrive/model/M30_hard_error_finetune/run_v1 \
  --num_classes 108 \
  --true_col mapped_label \
  --epochs 5 \
  --batch_size 32

/content/vaipe-contextual-pill-recognition
=== M30 HARD-ERROR CLASS-BALANCED FINETUNE ===
Train metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv
Val metadata  : /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv
Test metadata : /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv
M26 dir       : /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1
Output dir    : /content/drive/MyDrive/model/M30_hard_error_finetune/run_v1
Num classes   : 108
Pill backbone : tf_efficientnetv2_s.in21k_ft_in1k
Rx backbone   : resnet18.a1_in1k
Using device  : cuda | AMP: True

=== DETECTED COLUMNS ===
{
  "train_label_col": "mapped_label",
  "val_label_col": "mapped_label",
  "test_label_col": "mapped_label",
  "pill_col": "pill_crop_path",
  "prescription_col": "prescription_image_path",
  "context_col": "context_labels"
}

=== ROW COUNTS ===
Train ro

In [8]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m30_hard_error_finetune.py \
  --train_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv \
  --val_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv \
  --test_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv \
  --m26_dir /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1 \
  --output_dir /content/drive/MyDrive/model/M30_hard_error_finetune/run_v1 \
  --num_classes 108 \
  --true_col mapped_label \
  --epochs 0

/content/vaipe-contextual-pill-recognition
=== M30 HARD-ERROR CLASS-BALANCED FINETUNE ===
Train metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv
Val metadata  : /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv
Test metadata : /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv
M26 dir       : /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1
Output dir    : /content/drive/MyDrive/model/M30_hard_error_finetune/run_v1
Num classes   : 108
Pill backbone : tf_efficientnetv2_s.in21k_ft_in1k
Rx backbone   : resnet18.a1_in1k
Using device  : cuda | AMP: True

=== DETECTED COLUMNS ===
{
  "train_label_col": "mapped_label",
  "val_label_col": "mapped_label",
  "test_label_col": "mapped_label",
  "pill_col": "pill_crop_path",
  "prescription_col": "prescription_image_path",
  "context_col": "context_labels"
}

=== ROW COUNTS ===
Train ro

In [9]:
%cd /content/vaipe-contextual-pill-recognition

!sed -i 's/torch.load(best_path, map_location=device)/torch.load(best_path, map_location=device, weights_only=False)/g' train_m30_hard_error_finetune.py

/content/vaipe-contextual-pill-recognition


In [10]:
!grep -n "torch.load(best_path" train_m30_hard_error_finetune.py

832:    ckpt = torch.load(best_path, map_location=device, weights_only=False)


In [11]:
%cd /content/vaipe-contextual-pill-recognition

!python train_m30_hard_error_finetune.py \
  --train_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv \
  --val_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv \
  --test_metadata /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv \
  --m26_dir /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1 \
  --output_dir /content/drive/MyDrive/model/M30_hard_error_finetune/run_v1 \
  --num_classes 108 \
  --true_col mapped_label \
  --epochs 0

/content/vaipe-contextual-pill-recognition
=== M30 HARD-ERROR CLASS-BALANCED FINETUNE ===
Train metadata: /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_train_hard_metadata.csv
Val metadata  : /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_val_hard_metadata.csv
Test metadata : /content/drive/MyDrive/model/M24_metric_learning/hard_pair_metadata/m24_test_hard_metadata.csv
M26 dir       : /content/drive/MyDrive/model/M26_calibrated_context_ensemble/run_v1
Output dir    : /content/drive/MyDrive/model/M30_hard_error_finetune/run_v1
Num classes   : 108
Pill backbone : tf_efficientnetv2_s.in21k_ft_in1k
Rx backbone   : resnet18.a1_in1k
Using device  : cuda | AMP: True

=== DETECTED COLUMNS ===
{
  "train_label_col": "mapped_label",
  "val_label_col": "mapped_label",
  "test_label_col": "mapped_label",
  "pill_col": "pill_crop_path",
  "prescription_col": "prescription_image_path",
  "context_col": "context_labels"
}

=== ROW COUNTS ===
Train ro